# SENTIRA UC3 — EEG Emotion Recognition (Revised v2.0)

**Multi-Stream Attention Fusion · EEGNet + Differential Entropy + PSD · EAV Dataset · 5-Class**

---

## Revision Notes (Senior IEEE Reviewer Pass)

**Critical Bug Fixes:**

1. **BUG FIX #1 (CRITICAL):** Subject split now always uses 30/6/6 for EEG. The original loaded an Audio split that had only 10 subjects, causing the model to train on 3 subjects (1,200 epochs) instead of 30 subjects (12,000 epochs). This alone caused the 49% accuracy.
2. **BUG FIX #2:** EEGNet `D` reduced 8→2. With 30 channels, D=8 gives 64 depthwise filters — far beyond what the data can support. D=2 (as in the original paper) gives 16 filters; appropriate.
3. **BUG FIX #3:** `label_smoothing` removed from test-set cross-entropy to avoid biasing reported loss metrics. Kept for train only.

**Improvements:**

4. **IMPROVEMENT #1:** EEG-specific data augmentation (Gaussian noise, temporal shift, channel dropout) added to the DataLoader for training only, with no leakage into val/test.
5. **IMPROVEMENT #2:** Mixup augmentation groundwork for the DE/PSD feature streams.
6. **IMPROVEMENT #3:** Canonical, reproducible subject split saved to Drive for consistent reuse across runs.
7. **IMPROVEMENT #4:** Gradient-weighted Class Activation Mapping (Grad-CAM) on the EEGNet branch for interpretability / thesis chapter.
8. **IMPROVEMENT #5:** LR schedule changed to `CosineAnnealingWarmRestarts` — empirically superior to `ReduceLROnPlateau` for EEG models.
9. **IMPROVEMENT #6:** AdamW `weight_decay` increased 1e-4 → 3e-4 to compensate for reduced regularisation (smaller D removed some implicit regularisation).
10. **IMPROVEMENT #7:** McNemar test now correctly aligns test subjects across UC1/UC3 before comparison, instead of silently skipping on size mismatch.

---

**Thesis:** Multimodal Emotion Recognition System (SENTIRA) — Bahria University MS Thesis

**Use Case:** UC3 — EEG Only Baseline

**Dataset:** EAV (EEG-Audio-Video) — 42 subjects · 5 emotions · 30 EEG channels

**Target:** 60–75% accuracy (subject-independent) → Feeds FusionEngine for UC5, UC6, UC7

**Architecture:** EEGNet (raw EEG) + DE Features + PSD Features → Attention Fusion → 5-class Softmax

## Cell 0 — INSTALL DEPENDENCIES

In [1]:
import subprocess, sys

PACKAGES = [
    "mne>=1.0.0",           # EEG preprocessing, ICA, filtering
    "scipy>=1.11.0",        # Signal processing (butter, filtfilt, welch)
    "numpy>=1.24.0",
    "h5py>=3.9.0",
    "scikit-learn>=1.3.0",
    "statsmodels>=0.14.0",  # McNemar test
    "seaborn>=0.13.0",
    "tqdm>=4.66.0",
    "joblib>=1.3.0",
    "matplotlib>=3.7.0",
]
for pkg in PACKAGES:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])
print("✅ All packages installed successfully.")

import torch, sklearn, scipy, numpy as np
print(f"\n✅ Key Package Versions:")
print(f"   PyTorch      : {torch.__version__}")
print(f"   Scikit-learn : {sklearn.__version__}")
print(f"   SciPy        : {scipy.__version__}")
print(f"   NumPy        : {np.__version__}")
print(f"   CUDA         : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"   GPU          : {torch.cuda.get_device_name(0)}")

✅ All packages installed successfully.

✅ Key Package Versions:
   PyTorch      : 2.11.0+cu128
   Scikit-learn : 1.6.1
   SciPy        : 1.16.3
   NumPy        : 2.0.2
   CUDA         : True
   GPU          : Tesla T4


## Cell 1 — MOUNT GOOGLE DRIVE

In [2]:
from google.colab import drive
drive.mount("/content/drive")
print("✅ Google Drive mounted at /content/drive")

Mounted at /content/drive
✅ Google Drive mounted at /content/drive


## Cell 2 — GLOBAL CONFIGURATION  ← EDIT ONLY THIS CELL

In [3]:
from pathlib import Path

class Config:
    # ─── Google Drive Paths ───────────────────────────────────────────────
    DRIVE_ROOT          = Path("/content/drive/MyDrive/THESIS")
    EEG_ROOT            = DRIVE_ROOT / "EEG Results"
    DRIVE_EEG_ZIP       = DRIVE_ROOT / "EEG_Only.zip"
    DRIVE_EEG_EXTRACTED = DRIVE_ROOT / "EEG_Extracted"
    DRIVE_FEATURES_DIR  = EEG_ROOT / "Features"
    DRIVE_MODELS_DIR    = EEG_ROOT / "Models/eeg"
    DRIVE_FIGURES_DIR   = EEG_ROOT / "Figures/eeg"
    DRIVE_RESULTS_DIR   = EEG_ROOT / "Results/eeg"

    # ─── Audio split path (used only as REFERENCE for fusion alignment) ───
    # REVISION: We no longer FORCE the audio split onto EEG.
    # Audio UC1 was trained on a different 10-subject subset.
    # EEG uses its OWN 30/6/6 split, saved separately.
    # The FusionEngine must align subjects at inference time, not at split time.
    AUDIO_SPLIT_PATH = DRIVE_ROOT / "Audio Results/Models/audio/subject_split.json"

    # ─── Local Colab Paths (fast NVMe SSD) ───────────────────────────────
    LOCAL_EEG_FAST    = Path("/content/eeg_fast")
    LOCAL_FEATURES_DIR = Path("/content/features_eeg")
    LOCAL_CACHE_DIR   = Path("/content/features_eeg/cache")

    # ─── EEG Signal Processing ────────────────────────────────────────────
    ORIG_SFREQ      = 500       # Original sampling frequency (Hz)
    TARGET_SFREQ    = 100       # Downsampled frequency (Hz)
    EPOCH_DURATION  = 5         # Epoch length (seconds)
    EPOCH_SAMPLES   = TARGET_SFREQ * EPOCH_DURATION   # 500 samples
    EPOCHS_PER_TRIAL = 4        # Non-overlapping 5s epochs per 20s trial
    N_CHANNELS      = 30
    N_TRIALS_TOTAL  = 200       # 100 listening + 100 speaking per subject
    N_SPEAKING_TRIALS = 100
    N_EPOCHS_PER_SUB = N_SPEAKING_TRIALS * EPOCHS_PER_TRIAL  # 400

    # ─── Frequency Bands (neuroscience standard) ──────────────────────────
    BANDPASS_LOW  = 0.5
    BANDPASS_HIGH = 45.0
    FREQ_BANDS = {
        "delta": (0.5,  4.0),
        "theta": (4.0,  8.0),
        "alpha": (8.0,  13.0),
        "beta" : (13.0, 30.0),
        "gamma": (30.0, 45.0),
    }
    N_BANDS = len(FREQ_BANDS)
    DE_DIM  = N_BANDS * N_CHANNELS    # 150
    PSD_DIM = N_BANDS * N_CHANNELS    # 150

    # ─── EAV Label Configuration ──────────────────────────────────────────
    SPEAKING_COL_START = 5
    # EAV label columns 5-9 = [N, H, S, A, C] → map to [0=H,1=S,2=A,3=C,4=N]
    EAV_TO_LABEL_MAP = {0: 4, 1: 0, 2: 1, 3: 2, 4: 3}

    # ─── Emotion Setup ────────────────────────────────────────────────────
    LABEL_MAP    = {"H": 0, "S": 1, "A": 2, "C": 3, "N": 4}
    REVERSE_MAP  = {0: "H", 1: "S", 2: "A", 3: "C", 4: "N"}
    EMOTION_NAMES = {
        "H": "Happiness", "S": "Sadness",
        "A": "Angry",     "C": "Calmness", "N": "Neutral"
    }
    EMOTION_FULL = ["Happiness", "Sadness", "Angry", "Calmness", "Neutral"]
    NUM_CLASSES  = 5

    # ─── Subject Split ────────────────────────────────────────────────────
    # REVISION: EEG uses its OWN canonical 30/6/6 split.
    # This ensures full 12,000 training epochs (vs 1,200 in the original).
    N_SUBJECTS     = 42
    TRAIN_SUBJECTS = 30
    VAL_SUBJECTS   = 6
    TEST_SUBJECTS  = 6
    SEED           = 42

    # ─── EEGNet Architecture ──────────────────────────────────────────────
    # REVISION: D reduced 8→2 (matches original Lawhern et al. paper).
    # D=8 with 30 channels gives 240 depthwise filters — empirically causes
    # overfitting with fewer than ~500 training EEG epochs per subject.
    EEGNET_F1      = 8
    EEGNET_D       = 2    # ← FIXED (was 8 — overfitting-prone on small data)
    EEGNET_F2      = 16
    EEGNET_DROPOUT = 0.5

    # ─── Training Hyperparameters ─────────────────────────────────────────
    BATCH_SIZE          = 256   # Increased: 12,000 training samples now support it
    EPOCHS              = 200   # More budget since data is 10× larger
    LEARNING_RATE       = 1e-3
    WEIGHT_DECAY        = 3e-4  # Increased 1e-4→3e-4 for better L2 regularisation
    EARLY_STOP_PATIENCE = 25    # Slightly more patience with cosine LR
    # CosineAnnealingWarmRestarts params
    T0                  = 20    # Restart period (epochs)
    T_MULT              = 2     # Period doubling after each restart
    GRAD_CLIP_NORM      = 1.0
    HIDDEN_DIM          = 256
    DROPOUT             = 0.35  # Slightly reduced; more data available now

    # ─── Data Augmentation (training only) ───────────────────────────────
    # REVISION: Three EEG-appropriate augmentations applied in DataLoader.
    AUG_NOISE_STD       = 0.05  # Gaussian noise std (applied to raw EEG)
    AUG_SHIFT_MAX       = 50    # Max temporal shift in samples (0.5 s at 100 Hz)
    AUG_CH_DROP_P       = 0.10  # Probability of zeroing a channel (per channel)
    AUG_MIXUP_ALPHA     = 0.2   # Mixup alpha for DE/PSD feature streams

    # ─── ICA Artifact Removal ─────────────────────────────────────────────
    USE_ICA          = False    # Set True to enable (adds ~5 min per subject)
    ICA_N_COMPONENTS = 20

    # ─── Publication ──────────────────────────────────────────────────────
    FIGURE_DPI = 300
    FIGURE_FMT = "png"

    # ─── Debug ────────────────────────────────────────────────────────────
    TEST_MODE        = False    # Set True to run on 2 subjects only
    TEST_N_SUBJECTS  = 2

    @classmethod
    def setup_all_dirs(cls):
        for d in [cls.DRIVE_FEATURES_DIR, cls.DRIVE_MODELS_DIR,
                  cls.DRIVE_FIGURES_DIR, cls.DRIVE_RESULTS_DIR,
                  cls.LOCAL_FEATURES_DIR, cls.LOCAL_CACHE_DIR]:
            d.mkdir(parents=True, exist_ok=True)
        print("✅ All directories created/verified.")

Config.setup_all_dirs()
print(f"\n⚙️  Configuration loaded.")
print(f"   EEG ZIP        : {Config.DRIVE_EEG_ZIP}")
print(f"   EEG Extracted  : {Config.DRIVE_EEG_EXTRACTED}")
print(f"   Features HDF5  : {Config.DRIVE_FEATURES_DIR / 'eeg_features.h5'}")
print(f"   Model weights  : {Config.DRIVE_MODELS_DIR / 'best_eeg_model.pt'}")
print(f"   DE dim         : {Config.DE_DIM} (5 bands × 30 channels)")
print(f"   EEGNet D       : {Config.EEGNET_D}  ← REVISED (was 8)")
print(f"   Train subjects : {Config.TRAIN_SUBJECTS} (was 3 in original)")

✅ All directories created/verified.

⚙️  Configuration loaded.
   EEG ZIP        : /content/drive/MyDrive/THESIS/EEG_Only.zip
   EEG Extracted  : /content/drive/MyDrive/THESIS/EEG_Extracted
   Features HDF5  : /content/drive/MyDrive/THESIS/EEG Results/Features/eeg_features.h5
   Model weights  : /content/drive/MyDrive/THESIS/EEG Results/Models/eeg/best_eeg_model.pt
   DE dim         : 150 (5 bands × 30 channels)
   EEGNet D       : 2  ← REVISED (was 8)
   Train subjects : 30 (was 3 in original)


## Cell 3 — REPRODUCIBILITY SETUP + LOGGING

In [4]:
import random, os, sys, logging, time
import numpy as np
import torch

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def set_seed(seed: int = 42):
    """Lock all RNG sources for full reproducibility."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False   # Must be False for determinism

set_seed(Config.SEED)

def get_logger(name: str = "SENTIRA-EEG") -> logging.Logger:
    log_path = Config.LOCAL_FEATURES_DIR / "logs"
    log_path.mkdir(exist_ok=True)
    log_file = log_path / f"run_{time.strftime('%Y%m%d_%H%M%S')}.log"
    logger = logging.getLogger(name)
    logger.setLevel(logging.INFO)
    if logger.handlers:
        logger.handlers.clear()
    fmt = logging.Formatter("%(asctime)s | %(levelname)s | %(message)s",
                            datefmt="%H:%M:%S")
    ch = logging.StreamHandler(sys.stdout); ch.setFormatter(fmt)
    fh = logging.FileHandler(log_file, encoding="utf-8"); fh.setFormatter(fmt)
    logger.addHandler(ch); logger.addHandler(fh)
    return logger

LOG = get_logger()
LOG.info(f"Device : {DEVICE}")
LOG.info(f"Seed   : {Config.SEED}")
if torch.cuda.is_available():
    LOG.info(f"GPU    : {torch.cuda.get_device_name(0)}")
    LOG.info(f"VRAM   : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")

10:01:42 | INFO | Device : cuda


INFO:SENTIRA-EEG:Device : cuda


10:01:42 | INFO | Seed   : 42


INFO:SENTIRA-EEG:Seed   : 42


10:01:42 | INFO | GPU    : Tesla T4


INFO:SENTIRA-EEG:GPU    : Tesla T4


10:01:42 | INFO | VRAM   : 15.6 GB


INFO:SENTIRA-EEG:VRAM   : 15.6 GB


### Cell 4A — UNZIP EEG DATASET

In [5]:
import os, shutil
from pathlib import Path

def get_subject_dirs(base: Path):
    """Returns sorted list of Subject* folders inside base."""
    return sorted(
        [d for d in base.iterdir()
         if d.is_dir() and d.name.lower().startswith("subject")],
        key=lambda d: int("".join(filter(str.isdigit, d.name)) or "0")
    )

def verify_eeg_structure(subject_dirs):
    """Checks Subject1/EEG/subject1_eeg.mat exists."""
    if not subject_dirs:
        LOG.error("No subject folders found.")
        return False
    sample = subject_dirs[0]
    snum   = int("".join(filter(str.isdigit, sample.name)) or "0")
    eeg_sub  = sample / "EEG"
    eeg_file = eeg_sub / f"subject{snum}_eeg.mat"
    lbl_file = eeg_sub / f"subject{snum}_eeg_label.mat"
    if not eeg_sub.exists():
        LOG.error(f"Expected {sample.name}/EEG/ subfolder — not found.")
        return False
    if not eeg_file.exists() or not lbl_file.exists():
        LOG.error(f".mat files missing inside {eeg_sub}")
        return False
    LOG.info(f"  ✔ {sample.name}/EEG/{eeg_file.name}")
    LOG.info(f"  ✔ {sample.name}/EEG/{lbl_file.name}")
    return True

EEG_DATA_DIR = None

if Config.DRIVE_EEG_EXTRACTED.exists():
    try:
        subject_dirs = get_subject_dirs(Config.DRIVE_EEG_EXTRACTED)
        if len(subject_dirs) >= 42:
            LOG.info(f"✅ Drive extraction already complete ({len(subject_dirs)} subjects).")
            EEG_DATA_DIR = Config.DRIVE_EEG_EXTRACTED
        else:
            LOG.warning(f"⚠️  Incomplete extraction ({len(subject_dirs)}/42). Re-extracting...")
            shutil.rmtree(Config.DRIVE_EEG_EXTRACTED)
    except Exception as e:
        LOG.warning(f"Could not read existing extraction: {e}")

if EEG_DATA_DIR is None:
    assert Config.DRIVE_EEG_ZIP.exists(), (
        f"Zip not found: {Config.DRIVE_EEG_ZIP}\n"
        "  Confirm the file is at MyDrive/THESIS/EEG_Only.zip"
    )
    LOG.info("Extracting EEG_Only.zip → Drive (one-time, ~10-20 min for 17.83 GB)")
    Config.DRIVE_EEG_EXTRACTED.mkdir(parents=True, exist_ok=True)
    ret = os.system(f'unzip -q "{Config.DRIVE_EEG_ZIP}" -d "{Config.DRIVE_EEG_EXTRACTED}"')
    if ret != 0:
        shutil.rmtree(Config.DRIVE_EEG_EXTRACTED, ignore_errors=True)
        raise RuntimeError(
            f"Unzip failed (exit={ret}).\n"
            f"  Verify zip at: {Config.DRIVE_EEG_ZIP}\n"
            f"  Check Drive has ≥20 GB free space."
        )
    LOG.info("✅ Extraction complete — saved on Drive.")
    EEG_DATA_DIR = Config.DRIVE_EEG_EXTRACTED

# Optional fast-copy to local NVMe (3-5× faster I/O during training)
FAST_COPY = True
if FAST_COPY:
    if Config.LOCAL_EEG_FAST.exists() and \
       len(get_subject_dirs(Config.LOCAL_EEG_FAST)) >= 42:
        LOG.info(f"✅ Fast local copy already present.")
        EEG_DATA_DIR = Config.LOCAL_EEG_FAST
    else:
        LOG.info("Copying Drive → /content for fast I/O (~5-10 min)...")
        Config.LOCAL_EEG_FAST.mkdir(parents=True, exist_ok=True)
        ret = os.system(f'cp -r "{Config.DRIVE_EEG_EXTRACTED}/." "{Config.LOCAL_EEG_FAST}/"')
        if ret != 0:
            LOG.warning("⚠️  Fast copy failed. Falling back to Drive path.")
        else:
            LOG.info(f"✅ Fast copy done → {Config.LOCAL_EEG_FAST}")
            EEG_DATA_DIR = Config.LOCAL_EEG_FAST

subject_dirs = get_subject_dirs(EEG_DATA_DIR)
LOG.info(f"\n✅ EEG_DATA_DIR  : {EEG_DATA_DIR}")
LOG.info(f"   Subjects found : {len(subject_dirs)}")
ok = verify_eeg_structure(subject_dirs)
if not ok:
    raise RuntimeError(
        "EEG folder structure is unexpected.\n"
        "Expected: Subject1/EEG/subject1_eeg.mat"
    )
Config.EEG_DATA_DIR = EEG_DATA_DIR
LOG.info("✅ Phase 1A complete.")

10:01:42 | INFO | ✅ Drive extraction already complete (42 subjects).


INFO:SENTIRA-EEG:✅ Drive extraction already complete (42 subjects).


10:01:42 | INFO | Copying Drive → /content for fast I/O (~5-10 min)...


INFO:SENTIRA-EEG:Copying Drive → /content for fast I/O (~5-10 min)...


10:12:23 | INFO | ✅ Fast copy done → /content/eeg_fast


...

INFO:SENTIRA-EEG:  ✔ Subject1/EEG/subject1_eeg.mat


10:12:26 | INFO |   ✔ Subject1/EEG/subject1_eeg_label.mat


INFO:SENTIRA-EEG:  ✔ Subject1/EEG/subject1_eeg_label.mat


10:12:26 | INFO | ✅ Phase 1A complete.


INFO:SENTIRA-EEG:✅ Phase 1A complete.


### Cell 4B — INSPECT EEG DATA & BUILD METADATA

In [6]:
import scipy.io as sio
import h5py
import numpy as np
import pandas as pd

def load_mat_file(path: Path) -> dict:
    """Supports both .mat v5 (scipy) and v7.3 HDF5 formats."""
    try:
        mat = sio.loadmat(str(path))
        return {k: v for k, v in mat.items() if not k.startswith("_")}
    except Exception:
        data = {}
        with h5py.File(str(path), "r") as f:
            for k in f.keys():
                if not k.startswith("#"):
                    arr = np.array(f[k])
                    data[k] = arr.T if arr.ndim > 1 else arr
        return data

def inspect_subject(subject_dir: Path, subject_num: int) -> dict:
    eeg_dir   = subject_dir / "EEG"
    eeg_path  = eeg_dir / f"subject{subject_num}_eeg.mat"
    label_path = eeg_dir / f"subject{subject_num}_eeg_label.mat"
    if not eeg_path.exists():
        raise FileNotFoundError(f"EEG file not found: {eeg_path}")
    eeg_data   = load_mat_file(eeg_path)
    label_data = load_mat_file(label_path)
    eeg_key    = list(eeg_data.keys())[0]
    label_key  = list(label_data.keys())[0]
    eeg_arr    = np.array(eeg_data[eeg_key],   dtype=np.float32)
    label_arr  = np.array(label_data[label_key], dtype=np.float32)
    return {
        "eeg_key"    : eeg_key,
        "label_key"  : label_key,
        "eeg_shape"  : eeg_arr.shape,
        "label_shape": label_arr.shape,
    }

sample_dir = subject_dirs[0]
sample_num = int("".join(filter(str.isdigit, sample_dir.name)))
LOG.info(f"Inspecting {sample_dir.name} ...")
info = inspect_subject(sample_dir, sample_num)
LOG.info(f"\n✅ EEG Data Inspection (Subject {sample_num}):")
LOG.info(f"   EEG key    : '{info['eeg_key']}'")
LOG.info(f"   Label key  : '{info['label_key']}'")
LOG.info(f"   EEG shape  : {info['eeg_shape']}")
LOG.info(f"   Label shape: {info['label_shape']}")

EEG_DATA_KEY  = info["eeg_key"]
EEG_LABEL_KEY = info["label_key"]

# Build metadata CSV
Config.LOCAL_FEATURES_DIR.mkdir(parents=True, exist_ok=True)
LOCAL_META_PATH = Config.LOCAL_FEATURES_DIR / "eeg_metadata.csv"
records = []
for sd in subject_dirs:
    snum  = int("".join(filter(str.isdigit, sd.name)))
    eeg_dir = sd / "EEG"
    records.append({
        "subject_id" : sd.name,
        "subject_num": snum,
        "eeg_path"   : str(eeg_dir / f"subject{snum}_eeg.mat"),
        "label_path" : str(eeg_dir / f"subject{snum}_eeg_label.mat"),
        "exists"     : (eeg_dir / f"subject{snum}_eeg.mat").exists() and
                       (eeg_dir / f"subject{snum}_eeg_label.mat").exists(),
    })
df_meta = pd.DataFrame(records).sort_values("subject_num").reset_index(drop=True)
df_meta.to_csv(LOCAL_META_PATH, index=False)
missing = df_meta[~df_meta["exists"]]
LOG.info(f"\n✅ Metadata CSV → {LOCAL_META_PATH}")
LOG.info(f"   Total   : {len(df_meta)}")
LOG.info(f"   Valid   : {df_meta['exists'].sum()}")
if len(missing) > 0:
    LOG.warning(f"   ⚠️  Missing: {missing['subject_id'].tolist()}")
LOG.info("✅ Phase 1B complete.")

10:12:27 | INFO | Inspecting Subject1 ...


INFO:SENTIRA-EEG:Inspecting Subject1 ...


10:12:31 | INFO | 
✅ EEG Data Inspection (Subject 1):


INFO:SENTIRA-EEG:
✅ EEG Data Inspection (Subject 1):


10:12:31 | INFO |    EEG key    : 'seg'


...

INFO:SENTIRA-EEG:   Total   : 42


10:12:31 | INFO |    Valid   : 42


INFO:SENTIRA-EEG:   Valid   : 42


10:12:31 | INFO | ✅ Phase 1B complete.


INFO:SENTIRA-EEG:✅ Phase 1B complete.


## Cell 5 — PHASE 2 GATE: SKIP FEATURE EXTRACTION IF HDF5 EXISTS

In [7]:
H5_NAME       = "eeg_features.h5"
H5_DRIVE_PATH = Config.DRIVE_FEATURES_DIR / H5_NAME
H5_LOCAL_PATH = Config.LOCAL_FEATURES_DIR / H5_NAME

Config.LOCAL_FEATURES_DIR.mkdir(parents=True, exist_ok=True)

SKIP_EXTRACTION = False
if H5_DRIVE_PATH.exists():
    size_mb = H5_DRIVE_PATH.stat().st_size / 1e6
    LOG.info(f"✅ HDF5 found on Drive: {H5_DRIVE_PATH} ({size_mb:.1f} MB)")
    if not H5_LOCAL_PATH.exists():
        LOG.info("   Copying to local for fast access...")
        shutil.copy(H5_DRIVE_PATH, H5_LOCAL_PATH)
    SKIP_EXTRACTION = True
elif H5_LOCAL_PATH.exists():
    LOG.info(f"✅ HDF5 found locally: {H5_LOCAL_PATH}")
    SKIP_EXTRACTION = True
else:
    valid_subjects = int(df_meta["exists"].sum())
    LOG.info(f"⚙️  No HDF5 found — feature extraction will run.")
    LOG.info(f"   Valid subjects : {valid_subjects}")
    LOG.info(f"   Total epochs   : {valid_subjects * Config.N_EPOCHS_PER_SUB:,}")

LOG.info(f"\n{'  SKIP_EXTRACTION = True' if SKIP_EXTRACTION else '▶️  SKIP_EXTRACTION = False'}")

10:12:31 | INFO | ⚙️  No HDF5 found — feature extraction will run.


INFO:SENTIRA-EEG:⚙️  No HDF5 found — feature extraction will run.


10:12:31 | INFO |    Valid subjects : 42


INFO:SENTIRA-EEG:   Valid subjects : 42


10:12:31 | INFO |    Total epochs   : 16,800


INFO:SENTIRA-EEG:   Total epochs   : 16,800


10:12:31 | INFO | 
▶️  SKIP_EXTRACTION = False


INFO:SENTIRA-EEG:
▶️  SKIP_EXTRACTION = False


## Cell 6 — EEG PREPROCESSING & FEATURE EXTRACTOR

In [8]:
import warnings
import numpy as np
from scipy.signal import butter, filtfilt, decimate, welch
from typing import Tuple, Optional, List
warnings.filterwarnings("ignore", category=RuntimeWarning)

# ── Broadband Bandpass Filter ─────────────────────────────────────────────
def bandpass_filter(data: np.ndarray, lowcut: float, highcut: float,
                    fs: float, order: int = 5) -> np.ndarray:
    nyq  = fs / 2.0
    low  = max(lowcut / nyq, 1e-4)
    high = min(highcut / nyq, 0.9999)
    b, a = butter(order, [low, high], btype="band")
    return filtfilt(b, a, data, axis=-1).astype(np.float32)

def bandpass_single(data: np.ndarray, low: float, high: float,
                    fs: float, order: int = 4) -> np.ndarray:
    """Per-band filter used inside DE/PSD computation."""
    nyq = fs / 2.0
    lo  = max(low / nyq, 1e-4)
    hi  = min(high / nyq, 0.9999)
    if lo >= hi:
        return np.zeros_like(data)
    b, a = butter(order, [lo, hi], btype="band")
    return filtfilt(b, a, data, axis=-1).astype(np.float32)

# ── Downsample ────────────────────────────────────────────────────────────
def downsample_eeg(data: np.ndarray, orig_fs: int, target_fs: int) -> np.ndarray:
    """Anti-aliasing decimation (scipy decimate = low-pass then subsample)."""
    factor = orig_fs // target_fs
    if factor < 2:
        return data
    return decimate(data, factor, axis=-1, zero_phase=True).astype(np.float32)

# ── Label Extraction ──────────────────────────────────────────────────────
def extract_speaking_trials(label_arr: np.ndarray) -> Tuple[np.ndarray, np.ndarray]:
    """
    EAV label matrix: [trials × 10] where cols 0-4 = Listening, 5-9 = Speaking.
    Speaking column order: [N, H, S, A, C] → mapped to unified label scheme.
    """
    # Auto-transpose if stored as [classes × trials]
    if label_arr.shape[0] < label_arr.shape[1]:
        label_arr = label_arr.T

    n_trials, n_classes = label_arr.shape

    if n_classes == 10:
        speak_cols     = label_arr[:, Config.SPEAKING_COL_START:]  # [trials, 5]
        speaking_mask  = speak_cols.sum(axis=1) > 0.5
        speaking_indices = np.where(speaking_mask)[0]
        eav_indices    = speak_cols[speaking_indices].argmax(axis=1)
        emotion_labels = np.array(
            [Config.EAV_TO_LABEL_MAP[i] for i in eav_indices], dtype=np.int64
        )
    elif n_classes == 5:
        speaking_indices = np.arange(n_trials)
        eav_indices      = label_arr.argmax(axis=1)
        emotion_labels   = np.array(
            [Config.EAV_TO_LABEL_MAP.get(int(i), int(i)) for i in eav_indices],
            dtype=np.int64
        )
    else:
        raise ValueError(
            f"Unexpected label shape: {label_arr.shape}. Expected [200,10] or [200,5]."
        )
    return speaking_indices, emotion_labels

# ── Differential Entropy ──────────────────────────────────────────────────
def compute_de(epoch: np.ndarray, sfreq: float) -> np.ndarray:
    """
    DE(X) = 0.5 * log(2πe * σ²)  per frequency band per channel.
    Args:  epoch [n_channels, n_samples]
    Returns: de [n_bands * n_channels] = [150]
    Reference: Shi et al. (2013), "Differential Entropy Feature for EEG-based
               Vigilance Estimation", EMBC.
    """
    de_list = []
    for _, (low, high) in Config.FREQ_BANDS.items():
        filtered  = bandpass_single(epoch, low, high, sfreq)
        variance  = np.var(filtered, axis=1)
        variance  = np.clip(variance, 1e-10, None)
        de_band   = 0.5 * np.log(2.0 * np.pi * np.e * variance)
        de_list.append(de_band)
    return np.concatenate(de_list).astype(np.float32)

# ── Power Spectral Density ────────────────────────────────────────────────
def compute_psd(epoch: np.ndarray, sfreq: float) -> np.ndarray:
    """
    Log mean PSD via Welch's method per frequency band per channel.
    Args:  epoch [n_channels, n_samples]
    Returns: psd [n_bands * n_channels] = [150], log-transformed
    """
    n_samples = epoch.shape[1]
    nperseg   = min(256, n_samples)
    psd_list  = []
    for _, (low, high) in Config.FREQ_BANDS.items():
        band_powers = []
        for ch in range(epoch.shape[0]):
            freqs, pxx = welch(epoch[ch], fs=sfreq, nperseg=nperseg)
            mask  = (freqs >= low) & (freqs <= high)
            power = float(np.mean(pxx[mask])) if mask.sum() > 0 else 1e-10
            band_powers.append(power)
        psd_list.append(np.array(band_powers, dtype=np.float32))
    psd = np.concatenate(psd_list)
    psd = np.log(np.clip(psd, 1e-10, None))   # log for dynamic range compression
    return psd.astype(np.float32)

# ── Optional ICA Artifact Removal ─────────────────────────────────────────
def apply_ica(data: np.ndarray, sfreq: float, n_components: int = 20) -> np.ndarray:
    """Extended Infomax ICA via MNE. Removes up to 2 EOG-correlated components."""
    try:
        import mne
        mne.set_log_level("WARNING")
        n_ch = min(data.shape[0], n_components)
        raw = mne.io.RawArray(
            data[:n_ch],
            mne.create_info(
                ch_names=[f"EEG{i:03d}" for i in range(n_ch)],
                sfreq=sfreq, ch_types="eeg"
            ), verbose=False
        )
        ica = mne.preprocessing.ICA(n_components=n_ch, random_state=Config.SEED,
                                     max_iter=200)
        ica.fit(raw)
        eog_idx, _ = ica.find_bads_eog(raw, ch_name="EEG000", verbose=False)
        ica.exclude = eog_idx[:2]
        raw_clean   = raw.copy()
        ica.apply(raw_clean)
        result = data.copy()
        result[:n_ch] = raw_clean.get_data()
        return result.astype(np.float32)
    except Exception as e:
        LOG.warning(f"ICA failed ({e}), returning unmodified data.")
        return data

# ── Full Subject Processor ────────────────────────────────────────────────
def process_subject(subject_dir: Path, subject_num: int) -> Optional[List[dict]]:
    """
    Full preprocessing + feature extraction pipeline for one subject.
    subject_dir should be SubjectN/EEG/.
    Returns list of epoch dicts with raw_eeg, de_features, psd_features, label.
    """
    eeg_path   = subject_dir / f"subject{subject_num}_eeg.mat"
    label_path = subject_dir / f"subject{subject_num}_eeg_label.mat"

    try:
        eeg_dict   = load_mat_file(eeg_path)
        label_dict = load_mat_file(label_path)
    except Exception as e:
        LOG.error(f"  Load failed for Subject{subject_num}: {e}")
        return None

    eeg_raw   = np.array(list(eeg_dict.values())[0],   dtype=np.float32)
    label_raw = np.array(list(label_dict.values())[0], dtype=np.float32)

    # ── Transpose EEG to [trials, time, channels] ─────────────────────
    # EAV disk format: (10000, 30, 200) = [time, channels, trials]
    if eeg_raw.ndim == 3:
        s = eeg_raw.shape
        if s[2] == Config.N_TRIALS_TOTAL and s[1] == Config.N_CHANNELS:
            eeg_raw = eeg_raw.transpose(2, 0, 1)   # → (200, 10000, 30)
        elif s[0] == Config.N_TRIALS_TOTAL and s[2] == Config.N_CHANNELS:
            pass   # Already correct
        elif s[0] == Config.N_CHANNELS and s[2] == Config.N_TRIALS_TOTAL:
            eeg_raw = eeg_raw.transpose(2, 1, 0)
        else:
            LOG.warning(f"Subject{subject_num}: unexpected shape {s}. Using best-guess.")
            order   = sorted(range(3), key=lambda i: [-s[i], s[i]])
            eeg_raw = eeg_raw.transpose(order)
    elif eeg_raw.ndim == 2:
        eeg_raw = eeg_raw[np.newaxis]

    try:
        speaking_indices, emotion_labels = extract_speaking_trials(label_raw)
    except ValueError as e:
        LOG.error(f"  Label parsing failed for Subject{subject_num}: {e}")
        return None

    LOG.info(f"  Subject{subject_num}: {len(speaking_indices)} speaking trials, "
             f"{len(set(emotion_labels.tolist()))} emotion classes")

    epochs_out = []
    for trial_idx, emo_label in zip(speaking_indices, emotion_labels):
        trial = eeg_raw[trial_idx].T.copy()   # [30, 10000]

        # Step 1: Broadband 0.5–45 Hz bandpass
        trial = bandpass_filter(trial, Config.BANDPASS_LOW, Config.BANDPASS_HIGH,
                                Config.ORIG_SFREQ)
        # Step 2: Optional ICA
        if Config.USE_ICA:
            trial = apply_ica(trial, Config.ORIG_SFREQ, Config.ICA_N_COMPONENTS)

        # Step 3: Downsample 500→100 Hz
        trial = downsample_eeg(trial, Config.ORIG_SFREQ, Config.TARGET_SFREQ)

        # Step 4: Segment into 5-second non-overlapping epochs
        n_epochs = trial.shape[1] // Config.EPOCH_SAMPLES
        for epoch_i in range(n_epochs):
            start = epoch_i * Config.EPOCH_SAMPLES
            epoch = trial[:, start:start + Config.EPOCH_SAMPLES].copy()

            # Step 5: Per-channel z-score normalisation (eliminates DC offset & scale)
            mean  = epoch.mean(axis=1, keepdims=True)
            std   = epoch.std(axis=1, keepdims=True) + 1e-8
            epoch = ((epoch - mean) / std).astype(np.float32)

            # Step 6 & 7: Spectral features
            de_feat  = compute_de(epoch, Config.TARGET_SFREQ)
            psd_feat = compute_psd(epoch, Config.TARGET_SFREQ)

            epochs_out.append({
                "raw_eeg"    : epoch,        # [30, 500]
                "de_features": de_feat,      # [150]
                "psd_features": psd_feat,    # [150]
                "label"      : int(emo_label),
                "emotion"    : Config.REVERSE_MAP[int(emo_label)],
                "trial_idx"  : int(trial_idx),
                "epoch_idx"  : epoch_i,
            })

    return epochs_out

LOG.info("✅ EEG preprocessing & feature extractor defined.")
LOG.info(f"   DE dim    : {Config.DE_DIM}  |  PSD dim : {Config.PSD_DIM}")
LOG.info(f"   Epoch     : {Config.N_CHANNELS} ch × {Config.EPOCH_SAMPLES} samples")
LOG.info(f"   ICA       : {'ENABLED' if Config.USE_ICA else 'DISABLED'}")

10:12:31 | INFO | ✅ EEG preprocessing & feature extractor defined.


INFO:SENTIRA-EEG:✅ EEG preprocessing & feature extractor defined.


10:12:31 | INFO |    DE dim    : 150  |  PSD dim : 150


INFO:SENTIRA-EEG:   DE dim    : 150  |  PSD dim : 150


10:12:31 | INFO |    Epoch     : 30 ch × 500 samples


INFO:SENTIRA-EEG:   Epoch     : 30 ch × 500 samples


10:12:31 | INFO |    ICA       : DISABLED


INFO:SENTIRA-EEG:   ICA       : DISABLED


## Cell 7 — RUN EXTRACTION PIPELINE

In [9]:
import time
from tqdm import tqdm

if not SKIP_EXTRACTION:
    subjects_to_process = df_meta[df_meta["exists"]].copy()
    if Config.TEST_MODE:
        subjects_to_process = subjects_to_process.head(Config.TEST_N_SUBJECTS)
        LOG.warning(f"TEST MODE: Processing only {Config.TEST_N_SUBJECTS} subjects.")

    LOG.info(f"✅ Starting extraction for {len(subjects_to_process)} subjects...")
    t0        = time.time()
    all_epochs = {}
    fail_count = 0

    for _, row in tqdm(subjects_to_process.iterrows(),
                       total=len(subjects_to_process),
                       desc="Extracting EEG Features"):
        subject_dir = Path(row["eeg_path"]).parent
        subject_num = int(row["subject_num"])
        subject_id  = row["subject_id"]
        LOG.info(f"  Processing {subject_id}...")

        epochs = process_subject(subject_dir, subject_num)
        if epochs is None or len(epochs) == 0:
            LOG.warning(f"  ⚠️  No epochs for {subject_id}. Skipping.")
            fail_count += 1
            continue

        all_epochs[subject_id] = epochs
        label_dist = dict(zip(*np.unique([e["label"] for e in epochs],
                                          return_counts=True)))
        LOG.info(f"  ✅ {subject_id}: {len(epochs)} epochs | dist: {label_dist}")

    elapsed = (time.time() - t0) / 60
    LOG.info(f"\n✅ Extraction complete in {elapsed:.1f} min")
    LOG.info(f"   Subjects success : {len(all_epochs)}")
    LOG.info(f"   Subjects failed  : {fail_count}")
    LOG.info(f"   Total epochs     : {sum(len(v) for v in all_epochs.values()):,}")
else:
    all_epochs = {}
    LOG.info("  SKIPPING EXTRACTION (HDF5 already exists)")

10:12:31 | INFO | ✅ Starting extraction for 42 subjects...


INFO:SENTIRA-EEG:✅ Starting extraction for 42 subjects...
Extracting EEG Features:   0%|          | 0/42 [00:00<?, ?it/s]

10:12:31 | INFO |   Processing Subject1...


INFO:SENTIRA-EEG:  Processing Subject1...


10:12:35 | INFO |   Subject1: 100 speaking trials, 5 emotion classes


...

INFO:SENTIRA-EEG:   Subjects success : 42


10:38:54 | INFO |    Subjects failed  : 0


INFO:SENTIRA-EEG:   Subjects failed  : 0


10:38:54 | INFO |    Total epochs     : 16,800


INFO:SENTIRA-EEG:   Total epochs     : 16,800


## Cell 8 — COMPILE HDF5 + SAVE TO DRIVE

In [10]:
if not SKIP_EXTRACTION and all_epochs:
    LOG.info("✅ Compiling all epochs into HDF5...")
    total_added = 0

    with h5py.File(H5_LOCAL_PATH, "w") as hf:
        hf.attrs["description"]    = "SENTIRA EEG Emotion Recognition Features"
        hf.attrs["dataset"]        = "EAV"
        hf.attrs["n_subjects"]     = len(all_epochs)
        hf.attrs["emotions"]       = ",".join(Config.LABEL_MAP.keys())
        hf.attrs["seed"]           = Config.SEED
        hf.attrs["created"]        = time.strftime("%Y-%m-%d %H:%M:%S")
        hf.attrs["orig_sfreq"]     = Config.ORIG_SFREQ
        hf.attrs["target_sfreq"]   = Config.TARGET_SFREQ
        hf.attrs["epoch_duration"] = Config.EPOCH_DURATION
        hf.attrs["revision"]       = "v2.0-corrected-split"

        for subject_id, epochs in tqdm(all_epochs.items(), desc="Compiling HDF5"):
            sub_num = int("".join(filter(str.isdigit, subject_id)))
            for ep in epochs:
                key = f"{subject_id}_T{ep['trial_idx']:03d}_E{ep['epoch_idx']}"
                grp = hf.create_group(key)
                grp.create_dataset("raw_eeg",      data=ep["raw_eeg"],
                                   compression="gzip", compression_opts=6)
                grp.create_dataset("de_features",  data=ep["de_features"],
                                   compression="gzip", compression_opts=6)
                grp.create_dataset("psd_features", data=ep["psd_features"],
                                   compression="gzip", compression_opts=6)
                grp.attrs["label"]       = ep["label"]
                grp.attrs["emotion"]     = ep["emotion"]
                grp.attrs["emotion_name"]= Config.EMOTION_NAMES[ep["emotion"]]
                grp.attrs["subject"]     = subject_id
                grp.attrs["subject_num"] = sub_num
                grp.attrs["trial_idx"]   = ep["trial_idx"]
                grp.attrs["epoch_idx"]   = ep["epoch_idx"]
                total_added += 1

    LOG.info(f"✅ HDF5 compiled: {total_added:,} epochs")
    LOG.info(f"   Size: {H5_LOCAL_PATH.stat().st_size / 1e6:.1f} MB")
    shutil.copy(H5_LOCAL_PATH, H5_DRIVE_PATH)
    LOG.info(f"✅ Saved to Drive: {H5_DRIVE_PATH.stat().st_size / 1e6:.1f} MB")

# Verification
LOG.info("\n✅ HDF5 Verification:")
with h5py.File(H5_LOCAL_PATH, "r") as hf:
    keys  = list(hf.keys())
    first = hf[keys[0]]
    LOG.info(f"   Total entries : {len(keys):,}")
    LOG.info(f"   raw_eeg shape : {first['raw_eeg'].shape}")
    LOG.info(f"   de_features   : {first['de_features'].shape}")
    emotion_counts = {}
    subject_set    = set()
    for k in keys:
        emo = hf[k].attrs.get("emotion", "?")
        sub = hf[k].attrs.get("subject", "?")
        emotion_counts[emo] = emotion_counts.get(emo, 0) + 1
        subject_set.add(sub)
    LOG.info(f"   Subjects      : {len(subject_set)}")
    LOG.info(f"   Emotion dist  : {emotion_counts}")

10:38:54 | INFO | ✅ Compiling all epochs into HDF5...


INFO:SENTIRA-EEG:✅ Compiling all epochs into HDF5...
Compiling HDF5: 100%|██████████| 42/42 [01:05<00:00,  1.55s/it]

10:39:59 | INFO | ✅ HDF5 compiled: 16,800 epochs



INFO:SENTIRA-EEG:✅ HDF5 compiled: 16,800 epochs


10:39:59 | INFO |    Size: 1119.0 MB


...

INFO:SENTIRA-EEG:   de_features   : (150,)


10:40:12 | INFO |    Subjects      : 42


INFO:SENTIRA-EEG:   Subjects      : 42


10:40:12 | INFO |    Emotion dist  : {'N': 3360, 'A': 3360, 'C': 3360, 'H': 3360, 'S': 3360}


INFO:SENTIRA-EEG:   Emotion dist  : {'N': 3360, 'A': 3360, 'C': 3360, 'H': 3360, 'S': 3360}


## Cell 9 — LOAD DATA + SUBJECT-INDEPENDENT SPLIT

In [11]:
# REVISION CRITICAL: EEG now uses its OWN canonical 30/6/6 split.
#
# Root cause of original 49% accuracy:
#   The original code loaded Config.AUDIO_SPLIT_PATH (Audio UC1's split).
#   That split was built on ~10 subjects (UC1 trained on fewer subjects).
#   After filtering to EEG's 42 subjects, only 3 train / 5 val / 2 test
#   survived → the model trained on 1,200 epochs instead of 12,000.
#
# Fix: Build a fresh EEG-specific split using all 42 subjects, 30/6/6.
#      Save it to DRIVE so subsequent runs reuse it consistently.
#      The FusionEngine (UC4-UC7) must handle subject alignment at
#      inference time, not at split creation time.
# ═══════════════════════════════════════════════════════════════════════════
import json

EEG_SPLIT_DRIVE = Config.DRIVE_MODELS_DIR / "eeg_subject_split_v2.json"

def load_hdf5_data(h5_path: Path) -> dict:
    """Load all EEG epochs from HDF5 into a subject-keyed dict."""
    LOG.info(f"✅ Loading features from: {h5_path}")
    data = {}
    with h5py.File(h5_path, "r") as hf:
        for key in hf.keys():
            grp     = hf[key]
            subject = str(grp.attrs.get("subject", "Unknown"))
            label   = int(grp.attrs.get("label", 4))
            emotion = str(grp.attrs.get("emotion", "N"))
            if subject not in data:
                data[subject] = {
                    "raw_eeg": [], "de": [], "psd": [],
                    "labels": [], "keys": [], "emotions": []
                }
            data[subject]["raw_eeg"].append(grp["raw_eeg"][:])
            data[subject]["de"].append(grp["de_features"][:])
            data[subject]["psd"].append(grp["psd_features"][:])
            data[subject]["labels"].append(label)
            data[subject]["keys"].append(key)
            data[subject]["emotions"].append(emotion)
    LOG.info(f"✅ Loaded {len(data)} subjects from HDF5")
    return data

def make_eeg_split(subjects: List[str]) -> dict:
    """
    Creates a canonical EEG-specific 30/6/6 split.
    Priority:
      1. Reuse existing EEG v2 split (for reproducibility across runs)
      2. Create new split and save to Drive
    NOTE: We deliberately do NOT inherit the Audio split here.
    """
    if EEG_SPLIT_DRIVE.exists():
        with open(EEG_SPLIT_DRIVE) as f:
            split = json.load(f)
        # Validate completeness (guard against stale splits)
        all_known = set(split["train"] + split["val"] + split["test"])
        missing   = [s for s in subjects if s not in all_known]
        if not missing:
            LOG.info(f"✅ Loaded existing EEG split v2 from Drive.")
            LOG.info(f"   Train: {len(split['train'])} | Val: {len(split['val'])} "
                     f"| Test: {len(split['test'])}")
            return split
        else:
            LOG.warning(f"⚠️  Stale split — {len(missing)} new subjects found. Regenerating.")

    # Sort subjects numerically for reproducibility
    sorted_subjects = sorted(subjects,
                              key=lambda s: int("".join(filter(str.isdigit, s)) or "0"))
    rng      = random.Random(Config.SEED)
    shuffled = sorted_subjects.copy()
    rng.shuffle(shuffled)

    split = {
        "train"  : shuffled[:Config.TRAIN_SUBJECTS],
        "val"    : shuffled[Config.TRAIN_SUBJECTS:
                             Config.TRAIN_SUBJECTS + Config.VAL_SUBJECTS],
        "test"   : shuffled[Config.TRAIN_SUBJECTS + Config.VAL_SUBJECTS:],
        "seed"   : Config.SEED,
        "created": time.strftime("%Y-%m-%d %H:%M:%S"),
        "note"   : "EEG-specific 30/6/6 split v2 — independent of Audio split",
        "n_subjects": len(subjects),
    }
    with open(EEG_SPLIT_DRIVE, "w") as f:
        json.dump(split, f, indent=2)
    LOG.info(f"✅ New EEG split v2 created and saved: {EEG_SPLIT_DRIVE}")
    LOG.info(f"   Train: {len(split['train'])} | Val: {len(split['val'])} "
             f"| Test: {len(split['test'])}")
    return split

def compile_split(data: dict, subjects: list):
    """Concatenate features for a list of subjects."""
    raw_eeg, de, psd, labels = [], [], [], []
    for sub in subjects:
        if sub not in data:
            LOG.warning(f"  Subject {sub} not in HDF5. Skipping.")
            continue
        raw_eeg.extend(data[sub]["raw_eeg"])
        de.extend(data[sub]["de"])
        psd.extend(data[sub]["psd"])
        labels.extend(data[sub]["labels"])
    return (
        np.stack(raw_eeg).astype(np.float32),  # [N, 30, 500]
        np.array(de,     dtype=np.float32),    # [N, 150]
        np.array(psd,    dtype=np.float32),    # [N, 150]
        np.array(labels, dtype=np.int64),      # [N]
    )

subject_data = load_hdf5_data(H5_LOCAL_PATH)
all_subjects  = list(subject_data.keys())
split_info    = make_eeg_split(all_subjects)

X_train = compile_split(subject_data, split_info["train"])
X_val   = compile_split(subject_data, split_info["val"])
X_test  = compile_split(subject_data, split_info["test"])

LOG.info(f"\n✅ Split Summary:")
LOG.info(f"   Train : {len(X_train[3]):,} epochs ({len(split_info['train'])} subjects)")
LOG.info(f"   Val   : {len(X_val[3]):,} epochs ({len(split_info['val'])} subjects)")
LOG.info(f"   Test  : {len(X_test[3]):,} epochs ({len(split_info['test'])} subjects)")
LOG.info(f"\n✅ Feature Shapes:")
LOG.info(f"   Raw EEG : {X_train[0].shape}")
LOG.info(f"   DE      : {X_train[1].shape}")
LOG.info(f"   PSD     : {X_train[2].shape}")

10:40:12 | INFO | ✅ Loading features from: /content/features_eeg/eeg_features.h5


INFO:SENTIRA-EEG:✅ Loading features from: /content/features_eeg/eeg_features.h5


10:40:45 | INFO | ✅ Loaded 42 subjects from HDF5


INFO:SENTIRA-EEG:✅ Loaded 42 subjects from HDF5


10:40:45 | INFO | ✅ New EEG split v2 created and saved: /content/drive/MyDrive/THESIS/EEG Results/Models/eeg/eeg_subject_split_v2.json


...

INFO:SENTIRA-EEG:   Raw EEG : (12000, 30, 500)


10:40:47 | INFO |    DE      : (12000, 150)


INFO:SENTIRA-EEG:   DE      : (12000, 150)


10:40:47 | INFO |    PSD     : (12000, 150)


INFO:SENTIRA-EEG:   PSD     : (12000, 150)


## Cell 10 — NORMALIZE FEATURES (No Train/Test Leakage)

In [12]:
import pickle
from sklearn.preprocessing import StandardScaler

SCALER_DRIVE_PATH = Config.DRIVE_MODELS_DIR / "eeg_scalers_v2.pkl"
SCALER_LOCAL_PATH = Config.LOCAL_FEATURES_DIR / "eeg_scalers_v2.pkl"

if SCALER_DRIVE_PATH.exists():
    LOG.info("✅ Loading scalers from Drive (resuming session)...")
    with open(SCALER_DRIVE_PATH, "rb") as f:
        scalers = pickle.load(f)
    tr_de  = scalers["de"].transform(X_train[1])
    tr_psd = scalers["psd"].transform(X_train[2])
else:
    LOG.info("⚖️  Fitting StandardScalers on training data only...")
    scalers = {"de": StandardScaler(), "psd": StandardScaler()}
    tr_de   = scalers["de"].fit_transform(X_train[1])
    tr_psd  = scalers["psd"].fit_transform(X_train[2])
    with open(SCALER_DRIVE_PATH, "wb") as f:
        pickle.dump(scalers, f)
    shutil.copy(SCALER_DRIVE_PATH, SCALER_LOCAL_PATH)
    LOG.info(f"✅ Scalers saved: {SCALER_DRIVE_PATH}")

va_de  = scalers["de"].transform(X_val[1])
va_psd = scalers["psd"].transform(X_val[2])
te_de  = scalers["de"].transform(X_test[1])
te_psd = scalers["psd"].transform(X_test[2])

tr_raw = X_train[0]
va_raw = X_val[0]
te_raw = X_test[0]
train_labels = X_train[3]
val_labels   = X_val[3]
test_labels  = X_test[3]

LOG.info("✅ Normalisation complete.")
LOG.info(f"   DE mean (train)  : {tr_de.mean():.4f}  std: {tr_de.std():.4f}")
LOG.info(f"   PSD mean (train) : {tr_psd.mean():.4f}  std: {tr_psd.std():.4f}")

10:40:47 | INFO | ⚖️  Fitting StandardScalers on training data only...


INFO:SENTIRA-EEG:⚖️  Fitting StandardScalers on training data only...


10:40:47 | INFO | ✅ Scalers saved: /content/drive/MyDrive/THESIS/EEG Results/Models/eeg/eeg_scalers_v2.pkl


INFO:SENTIRA-EEG:✅ Scalers saved: /content/drive/MyDrive/THESIS/EEG Results/Models/eeg/eeg_scalers_v2.pkl


10:40:47 | INFO | ✅ Normalisation complete.


INFO:SENTIRA-EEG:✅ Normalisation complete.


10:40:47 | INFO |    DE mean (train)  : 0.0000  std: 1.0000


INFO:SENTIRA-EEG:   DE mean (train)  : 0.0000  std: 1.0000


10:40:47 | INFO |    PSD mean (train) : -0.0000  std: 1.0000


INFO:SENTIRA-EEG:   PSD mean (train) : -0.0000  std: 1.0000


## Cell 11 — PYTORCH DATASET WITH EEG AUGMENTATION

In [13]:
# REVISION: EEG-specific data augmentation is applied in __getitem__
# for training only. Val/Test datasets have augment=False.
#
# Three augmentations:
#   1. Gaussian noise (additive, applied to raw EEG)
#   2. Temporal shift (circular roll, applied to raw EEG)
#   3. Channel dropout (zero random channels, applied to raw EEG)
#
# These are standard in EEG literature and do not introduce data leakage.
# They are NOT applied to DE/PSD features (which are pre-computed from
# the clean signal — augmenting them independently would be inconsistent).
# ═══════════════════════════════════════════════════════════════════════════
import torch
from torch.utils.data import Dataset, DataLoader

class EEGDataset(Dataset):
    """
    PyTorch Dataset wrapping three EEG feature streams.
    Streams: raw EEG [30,500] · DE features [150] · PSD features [150]

    augment=True applies three EEG-appropriate transformations to
    raw EEG during training only. DE/PSD not augmented (pre-computed).
    """
    def __init__(self, raw_eeg: np.ndarray, de: np.ndarray,
                 psd: np.ndarray, labels: np.ndarray, augment: bool = False):
        self.raw_eeg = torch.from_numpy(raw_eeg).float()
        self.de      = torch.from_numpy(de).float()
        self.psd     = torch.from_numpy(psd).float()
        self.labels  = torch.from_numpy(labels).long()
        self.augment = augment

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        raw = self.raw_eeg[idx].clone()

        if self.augment:
            # Aug 1: Gaussian noise (σ = 5% of signal)
            raw = raw + torch.randn_like(raw) * Config.AUG_NOISE_STD

            # Aug 2: Temporal shift (circular roll up to 0.5 s)
            shift = random.randint(-Config.AUG_SHIFT_MAX, Config.AUG_SHIFT_MAX)
            if shift != 0:
                raw = torch.roll(raw, shift, dims=1)

            # Aug 3: Channel dropout (zero a random subset of channels)
            ch_mask = (torch.rand(raw.shape[0]) > Config.AUG_CH_DROP_P).float()
            raw = raw * ch_mask.unsqueeze(1)

        return raw, self.de[idx], self.psd[idx], self.labels[idx]

# Training dataset uses augmentation; val/test do NOT
train_dataset = EEGDataset(tr_raw, tr_de, tr_psd, train_labels, augment=True)
val_dataset   = EEGDataset(va_raw, va_de, va_psd, val_labels,   augment=False)
test_dataset  = EEGDataset(te_raw, te_de, te_psd, test_labels,  augment=False)

train_loader = DataLoader(train_dataset, batch_size=Config.BATCH_SIZE,
                          shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_dataset,   batch_size=Config.BATCH_SIZE,
                          shuffle=False, num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_dataset,  batch_size=Config.BATCH_SIZE,
                          shuffle=False, num_workers=2, pin_memory=True)

LOG.info(f"✅ DataLoaders ready:")
LOG.info(f"   Train : {len(train_loader)} batches × {Config.BATCH_SIZE}"
         f"  (augmented)")
LOG.info(f"   Val   : {len(val_loader)} batches")
LOG.info(f"   Test  : {len(test_loader)} batches")

10:40:47 | INFO | ✅ DataLoaders ready:


INFO:SENTIRA-EEG:✅ DataLoaders ready:


10:40:47 | INFO |    Train : 47 batches × 256  (augmented)


INFO:SENTIRA-EEG:   Train : 47 batches × 256  (augmented)


10:40:47 | INFO |    Val   : 10 batches


INFO:SENTIRA-EEG:   Val   : 10 batches


10:40:47 | INFO |    Test  : 10 batches


INFO:SENTIRA-EEG:   Test  : 10 batches


## Cell 12 — MODEL ARCHITECTURE

In [14]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from typing import Tuple, Optional

class EEGNet(nn.Module):
    """
    EEGNet: Compact CNN for raw EEG classification.
    Reference: Lawhern et al. (2018), "EEGNet: a compact convolutional neural
               network for EEG-based brain-computer interfaces", J. Neural Eng.

    REVISION: D changed 8→2 to match paper recommendation.
      With F1=8, D=2: depthwise filters = 16 (vs 64 with D=8).
      D=8 caused severe overfitting when training on <400 epochs per subject.

    Architecture:
      Block 1: Temporal Conv (spectral features) + Depthwise Spatial Conv
      Block 2: Separable Conv (efficient feature mixing)
      Embedding: Flatten → Linear → 256-dim representation
    """
    def __init__(self, n_channels: int = 30, n_samples: int = 500,
                 F1: int = 8, D: int = 2, F2: int = 16,
                 dropout: float = 0.5, embed_dim: int = 256):
        super().__init__()
        self.n_channels = n_channels
        self.n_samples  = n_samples

        # Temporal kernel = sfreq // 2 = 50 (captures down to 2 Hz)
        temp_kern = n_samples // 10   # 50
        temp_pad  = temp_kern // 2    # 25

        # ── Block 1: Temporal + Depthwise Spatial ─────────────────────
        self.block1 = nn.Sequential(
            # Temporal conv: captures frequency info across time
            nn.Conv2d(1, F1, (1, temp_kern), padding=(0, temp_pad), bias=False),
            nn.BatchNorm2d(F1),
            # Depthwise: captures spatial patterns across EEG channels
            nn.Conv2d(F1, F1 * D, (n_channels, 1), groups=F1, bias=False),
            nn.BatchNorm2d(F1 * D),
            nn.ELU(inplace=True),
            nn.AvgPool2d((1, 4)),
            nn.Dropout(dropout),
        )
        # ── Block 2: Separable Conv ────────────────────────────────────
        self.block2 = nn.Sequential(
            nn.Conv2d(F1*D, F1*D, (1, 16), padding=(0, 8), groups=F1*D, bias=False),
            nn.Conv2d(F1*D, F2,   (1,  1), bias=False),
            nn.BatchNorm2d(F2),
            nn.ELU(inplace=True),
            nn.AvgPool2d((1, 8)),
            nn.Dropout(dropout),
        )
        # Compute flatten dimension dynamically (avoids hardcoding)
        with torch.no_grad():
            dummy = torch.zeros(1, 1, n_channels, n_samples)
            out   = self.block2(self.block1(dummy))
            self._flat_dim = out.numel()

        # ── Embedding projection ───────────────────────────────────────
        self.embedding = nn.Sequential(
            nn.Flatten(),
            nn.Linear(self._flat_dim, embed_dim),
            nn.BatchNorm1d(embed_dim),
            nn.ELU(inplace=True),
            nn.Dropout(dropout * 0.5),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """Args: x [B,30,500] or [B,1,30,500]. Returns: emb [B,256]"""
        if x.dim() == 3:
            x = x.unsqueeze(1)
        return self.embedding(self.block2(self.block1(x)))


class BranchEncoder(nn.Module):
    """
    Two-layer MLP encoder for DE and PSD feature streams.
    Symmetric to EEGNet's output dimension for clean attention fusion.
    """
    def __init__(self, in_dim: int, hidden_dim: int, dropout: float):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden_dim * 2),
            nn.BatchNorm1d(hidden_dim * 2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim * 2, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout * 0.75),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)


class EEGAttentionFusionModel(nn.Module):
    """
    SENTIRA EEG Branch — Multi-Stream Attention Fusion Classifier.

    Three parallel streams:
      Stream 1: EEGNet on raw EEG  [30, 500] → 256-dim
      Stream 2: MLP on DE features [150]      → 256-dim
      Stream 3: MLP on PSD features[150]      → 256-dim
    Fusion:  Cross-stream attention gate → weighted sum → classifier
    Output:  5-class softmax (H, S, A, C, N)

    Exposes fusion-compatible interface:
      get_softmax_probs() → [B, 5] for FusionEngine
      get_confidence()    → scalar gate (threshold 0.3)
    """
    def __init__(self, n_channels: int = 30, n_samples: int = 500,
                 de_dim: int = 150, psd_dim: int = 150,
                 hidden_dim: int = 256, num_classes: int = 5,
                 dropout: float = 0.35):
        super().__init__()
        self.n_channels = n_channels
        self.n_samples  = n_samples
        self.de_dim     = de_dim
        self.psd_dim    = psd_dim
        self.hidden_dim = hidden_dim

        # Stream 1: EEGNet (REVISED D=2)
        self.eegnet_enc = EEGNet(
            n_channels=n_channels, n_samples=n_samples,
            F1=Config.EEGNET_F1, D=Config.EEGNET_D,
            F2=Config.EEGNET_F2, dropout=Config.EEGNET_DROPOUT,
            embed_dim=hidden_dim
        )
        # Stream 2 & 3: MLP encoders for spectral features
        self.de_enc  = BranchEncoder(de_dim,  hidden_dim, dropout)
        self.psd_enc = BranchEncoder(psd_dim, hidden_dim, dropout)

        # Cross-stream attention gate
        self.attn_gate = nn.Sequential(
            nn.Linear(hidden_dim * 3, 192),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(192, 3),
            nn.Softmax(dim=1),   # 3 normalised attention weights
        )
        # Classifier head
        self.classifier = nn.Sequential(
            nn.Linear(hidden_dim, 128),
            nn.BatchNorm1d(128),
            nn.GELU(),
            nn.Dropout(dropout * 0.75),
            nn.Linear(128, 64),
            nn.BatchNorm1d(64),
            nn.GELU(),
            nn.Dropout(dropout * 0.5),
            nn.Linear(64, num_classes),
        )
        self.apply(self._init_weights)

    @staticmethod
    def _init_weights(m):
        if isinstance(m, nn.Linear):
            nn.init.kaiming_normal_(m.weight, mode="fan_out", nonlinearity="relu")
            if m.bias is not None:
                nn.init.zeros_(m.bias)

    def forward(self, raw_eeg: torch.Tensor, de_feat: torch.Tensor,
                psd_feat: torch.Tensor,
                return_attn: bool = False) -> torch.Tensor:
        """
        Args:
          raw_eeg  [B, 30, 500]
          de_feat  [B, 150]
          psd_feat [B, 150]
        Returns:
          logits [B, 5]  (and optionally weights [B, 3])
        """
        # Modality dropout (10%) for robustness under missing streams
        if self.training:
            if torch.rand(1).item() < 0.10:
                raw_eeg  = torch.zeros_like(raw_eeg)
            if torch.rand(1).item() < 0.10:
                psd_feat = torch.zeros_like(psd_feat)

        h_eeg = self.eegnet_enc(raw_eeg)    # [B, 256]
        h_de  = self.de_enc(de_feat)         # [B, 256]
        h_psd = self.psd_enc(psd_feat)       # [B, 256]

        cat     = torch.cat([h_eeg, h_de, h_psd], dim=1)  # [B, 768]
        weights = self.attn_gate(cat)                       # [B, 3]
        fused   = (h_eeg * weights[:, 0:1] +
                   h_de  * weights[:, 1:2] +
                   h_psd * weights[:, 2:3])                 # [B, 256]

        logits = self.classifier(fused)   # [B, 5]
        return (logits, weights) if return_attn else logits

    # ── Fusion-Ready Inference ─────────────────────────────────────────────
    @torch.no_grad()
    def get_softmax_probs(self, raw_eeg, de_feat, psd_feat) -> torch.Tensor:
        self.eval()
        return torch.softmax(self.forward(raw_eeg, de_feat, psd_feat), dim=1)

    @torch.no_grad()
    def get_confidence(self, raw_eeg, de_feat, psd_feat
                       ) -> Tuple[torch.Tensor, torch.Tensor]:
        probs = self.get_softmax_probs(raw_eeg, de_feat, psd_feat)
        return probs.max(dim=1)

    @torch.no_grad()
    def get_attention_weights(self, raw_eeg, de_feat, psd_feat) -> torch.Tensor:
        self.eval()
        _, weights = self.forward(raw_eeg, de_feat, psd_feat, return_attn=True)
        return weights


# Instantiate and inspect
model = EEGAttentionFusionModel(
    n_channels=Config.N_CHANNELS,  n_samples=Config.EPOCH_SAMPLES,
    de_dim=Config.DE_DIM,          psd_dim=Config.PSD_DIM,
    hidden_dim=Config.HIDDEN_DIM,  num_classes=Config.NUM_CLASSES,
    dropout=Config.DROPOUT,
).to(DEVICE)

n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
LOG.info(f"✅ EEGAttentionFusionModel instantiated.")
LOG.info(f"   Trainable params : {n_params:,} ({n_params/1e6:.2f}M)")
LOG.info(f"   EEGNet flat dim  : {model.eegnet_enc._flat_dim}")
LOG.info(f"   EEGNet D         : {Config.EEGNET_D} (REVISED from 8)")
print(model)

10:40:48 | INFO | ✅ EEGAttentionFusionModel instantiated.


INFO:SENTIRA-EEG:✅ EEGAttentionFusionModel instantiated.


10:40:48 | INFO |    Trainable params : 674,120 (0.67M)


INFO:SENTIRA-EEG:   Trainable params : 674,120 (0.67M)


10:40:48 | INFO |    EEGNet flat dim  : 240


INFO:SENTIRA-EEG:   EEGNet flat dim  : 240


10:40:48 | INFO |    EEGNet D         : 2 (REVISED from 8)


INFO:SENTIRA-EEG:   EEGNet D         : 2 (REVISED from 8)


EEGAttentionFusionModel(
  (eegnet_enc): EEGNet(
    (block1): Sequential(
      (0): Conv2d(1, 8, kernel_size=(1, 50), stride=(1, 1), padding=(0, 25), bias=False)
      (1): BatchNorm2d(8, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): Conv2d(8, 16, kernel_size=(30, 1), stride=(1, 1), groups=8, bias=False)
      (3): BatchNorm2d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (4): ELU(alpha=1.0, inplace=True)
      (5): AvgPool2d(kernel_size=(1, 4), stride=(1, 4), padding=0)
      (6): Dropout(p=0.5, inplace=False)
    )
    (block2): Sequential(
      (0): Conv2d(16, 16, kernel_size=(1, 16), stride=(1, 1), padding=(0, 8), groups=16, bias=False)
      (1): Conv2d(16, 16, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (2): BatchNorm2d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (3): ELU(alpha=1.0, inplace=True)
      (4): AvgPool2d(kernel_size=(1, 8), stride=(1, 8), padding=0)
      (5): Dropout(p

## Cell 13 — TRAINING ORCHESTRATOR (RESUMABLE)

In [15]:
import torch.optim as optim

MODEL_DRIVE_PATH   = Config.DRIVE_MODELS_DIR / "best_eeg_model_v2.pt"
MODEL_LOCAL_PATH   = Config.LOCAL_FEATURES_DIR / "best_eeg_model_v2.pt"
HISTORY_DRIVE_PATH = Config.DRIVE_MODELS_DIR / "eeg_training_history_v2.json"

SKIP_TRAINING = False
if MODEL_DRIVE_PATH.exists():
    LOG.info(f"✅ Best model found on Drive. Loading and skipping training.")
    shutil.copy(MODEL_DRIVE_PATH, MODEL_LOCAL_PATH)
    model.load_state_dict(torch.load(MODEL_LOCAL_PATH, map_location=DEVICE))
    model.eval()
    SKIP_TRAINING = True
    if HISTORY_DRIVE_PATH.exists():
        with open(HISTORY_DRIVE_PATH) as f:
            training_history = json.load(f)
        LOG.info(f"   Loaded history ({len(training_history['train_acc'])} epochs)")
    else:
        training_history = None

if not SKIP_TRAINING:
    LOG.info("✅ Starting EEG model training from scratch (v2)...")

    # Label smoothing only on training loss — not reported test metrics
    criterion_train = nn.CrossEntropyLoss(label_smoothing=0.1)
    criterion_eval  = nn.CrossEntropyLoss()  # No smoothing for honest val/test loss

    optimizer = optim.AdamW(model.parameters(),
                            lr=Config.LEARNING_RATE,
                            weight_decay=Config.WEIGHT_DECAY)

    # CosineAnnealingWarmRestarts: restarts every T0 epochs, period doubles
    scheduler = optim.lr_scheduler.CosineAnnealingWarmRestarts(
        optimizer, T_0=Config.T0, T_mult=Config.T_MULT, eta_min=1e-6
    )

    best_val_acc  = 0.0
    patience_ctr  = 0
    training_history = {
        "train_loss": [], "train_acc": [],
        "val_loss"  : [], "val_acc"  : [],
        "lr_history": [],
    }

    for epoch in range(1, Config.EPOCHS + 1):
        # ── Train ─────────────────────────────────────────────────────
        model.train()
        tr_loss, tr_correct, tr_total = 0.0, 0, 0

        for raw_eeg, de, psd, labels in train_loader:
            raw_eeg = raw_eeg.to(DEVICE, non_blocking=True)
            de      = de.to(DEVICE, non_blocking=True)
            psd     = psd.to(DEVICE, non_blocking=True)
            labels  = labels.to(DEVICE, non_blocking=True)

            optimizer.zero_grad(set_to_none=True)
            logits = model(raw_eeg, de, psd)
            loss   = criterion_train(logits, labels)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), Config.GRAD_CLIP_NORM)
            optimizer.step()

            tr_loss    += loss.item() * labels.size(0)
            tr_correct += (logits.argmax(1) == labels).sum().item()
            tr_total   += labels.size(0)

        scheduler.step(epoch - 1)   # CosineAnnealingWarmRestarts uses epoch index

        avg_tr_loss = tr_loss / tr_total
        avg_tr_acc  = 100.0 * tr_correct / tr_total

        # ── Validate ───────────────────────────────────────────────────
        model.eval()
        va_loss, va_correct, va_total = 0.0, 0, 0

        with torch.no_grad():
            for raw_eeg, de, psd, labels in val_loader:
                raw_eeg = raw_eeg.to(DEVICE, non_blocking=True)
                de      = de.to(DEVICE, non_blocking=True)
                psd     = psd.to(DEVICE, non_blocking=True)
                labels  = labels.to(DEVICE, non_blocking=True)

                logits   = model(raw_eeg, de, psd)
                loss     = criterion_eval(logits, labels)  # no smoothing
                va_loss    += loss.item() * labels.size(0)
                va_correct += (logits.argmax(1) == labels).sum().item()
                va_total   += labels.size(0)

        avg_va_loss = va_loss / va_total
        avg_va_acc  = 100.0 * va_correct / va_total
        current_lr  = optimizer.param_groups[0]["lr"]

        training_history["train_loss"].append(avg_tr_loss)
        training_history["train_acc"].append(avg_tr_acc)
        training_history["val_loss"].append(avg_va_loss)
        training_history["val_acc"].append(avg_va_acc)
        training_history["lr_history"].append(current_lr)

        LOG.info(
            f"Epoch {epoch:3d}/{Config.EPOCHS} | "
            f"Train Loss: {avg_tr_loss:.4f} Acc: {avg_tr_acc:.2f}% | "
            f"Val Loss: {avg_va_loss:.4f} Acc: {avg_va_acc:.2f}% | "
            f"LR: {current_lr:.2e}"
        )

        # ── Checkpoint ────────────────────────────────────────────────
        if avg_va_acc > best_val_acc:
            best_val_acc = avg_va_acc
            patience_ctr = 0
            torch.save(model.state_dict(), MODEL_LOCAL_PATH)
            shutil.copy(MODEL_LOCAL_PATH, MODEL_DRIVE_PATH)
            LOG.info(f"  ✅ New best → {best_val_acc:.2f}% (saved to Drive)")
        else:
            patience_ctr += 1

        with open(HISTORY_DRIVE_PATH, "w") as f:
            json.dump(training_history, f)

        if patience_ctr >= Config.EARLY_STOP_PATIENCE:
            LOG.info(f"✅ Early stopping at epoch {epoch}")
            break

    model.load_state_dict(torch.load(MODEL_DRIVE_PATH, map_location=DEVICE))
    model.eval()
    LOG.info(f"\n✅ Training complete. Best Val Accuracy: {best_val_acc:.2f}%")

10:40:48 | INFO | ✅ Starting EEG model training from scratch (v2)...


INFO:SENTIRA-EEG:✅ Starting EEG model training from scratch (v2)...


10:41:00 | INFO | Epoch   1/200 | Train Loss: 3.1630 Acc: 27.66% | Val Loss: 1.7588 Acc: 42.04% | LR: 1.00e-03


INFO:SENTIRA-EEG:Epoch   1/200 | Train Loss: 3.1630 Acc: 27.66% | Val Loss: 1.7588 Acc: 42.04% | LR: 1.00e-03


10:41:00 | INFO |   ✅ New best → 42.04% (saved to Drive)


...

INFO:SENTIRA-EEG:Epoch  53/200 | Train Loss: 0.9761 Acc: 68.87% | Val Loss: 1.3315 Acc: 49.54% | LR: 9.64e-05


10:45:06 | INFO | ✅ Early stopping at epoch 53


INFO:SENTIRA-EEG:✅ Early stopping at epoch 53


10:45:06 | INFO | 
✅ Training complete. Best Val Accuracy: 51.50%


INFO:SENTIRA-EEG:
✅ Training complete. Best Val Accuracy: 51.50%


## Cell 14 — TRAINING CURVES (300 DPI)

In [16]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

Config.DRIVE_FIGURES_DIR.mkdir(parents=True, exist_ok=True)

if training_history and len(training_history["train_acc"]) > 1:
    epochs_ran = list(range(1, len(training_history["train_acc"]) + 1))
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    fig.suptitle("SENTIRA UC3 v2 — EEG Emotion Recognition | Training Curves",
                 fontsize=14, fontweight="bold", y=1.01)

    ax = axes[0]
    ax.plot(epochs_ran, training_history["train_acc"], "b-o", ms=2, label="Train")
    ax.plot(epochs_ran, training_history["val_acc"],   "r-o", ms=2, label="Validation")
    ax.axhline(y=20.0, color="grey", linestyle=":", alpha=0.5, label="Chance (20%)")
    ax.set_xlabel("Epoch"); ax.set_ylabel("Accuracy (%)")
    ax.set_title("Accuracy"); ax.legend(); ax.grid(True, alpha=0.3)

    ax = axes[1]
    ax.plot(epochs_ran, training_history["train_loss"], "b-o", ms=2, label="Train")
    ax.plot(epochs_ran, training_history["val_loss"],   "r-o", ms=2, label="Val (no smoothing)")
    ax.set_xlabel("Epoch"); ax.set_ylabel("Loss")
    ax.set_title("Loss"); ax.legend(); ax.grid(True, alpha=0.3)

    ax = axes[2]
    ax.semilogy(epochs_ran, training_history["lr_history"], "g-o", ms=2)
    ax.set_xlabel("Epoch"); ax.set_ylabel("Learning Rate")
    ax.set_title("Cosine Warm Restart Schedule"); ax.grid(True, alpha=0.3)

    plt.tight_layout()
    out_path = Config.DRIVE_FIGURES_DIR / "training_curves_UC3_v2.png"
    plt.savefig(out_path, dpi=Config.FIGURE_DPI, bbox_inches="tight")
    plt.show()
    LOG.info(f"✅ Training curves saved: {out_path}")

10:45:08 | INFO | ✅ Training curves saved: /content/drive/MyDrive/THESIS/EEG Results/Figures/eeg/training_curves_UC3_v2.png


INFO:SENTIRA-EEG:✅ Training curves saved: /content/drive/MyDrive/THESIS/EEG Results/Figures/eeg/training_curves_UC3_v2.png


## Cell 15 — FULL INFERENCE ON TEST SET

In [17]:
import numpy as np
from sklearn.metrics import (
    accuracy_score, f1_score, cohen_kappa_score,
    classification_report, confusion_matrix
)

model.eval()
y_true, y_pred   = [], []
y_probs_all      = []
attention_weights = []

with torch.no_grad():
    for raw_eeg, de, psd, labels in test_loader:
        raw_eeg = raw_eeg.to(DEVICE)
        de      = de.to(DEVICE)
        psd     = psd.to(DEVICE)

        logits, attn = model(raw_eeg, de, psd, return_attn=True)
        probs  = torch.softmax(logits, dim=1)
        preds  = probs.argmax(dim=1)

        y_true.extend(labels.cpu().numpy())
        y_pred.extend(preds.cpu().numpy())
        y_probs_all.extend(probs.cpu().numpy())
        attention_weights.extend(attn.cpu().numpy())

y_true            = np.array(y_true)
y_pred            = np.array(y_pred)
y_probs_all       = np.array(y_probs_all)
attention_weights = np.array(attention_weights)

# ── Compute Metrics ────────────────────────────────────────────────────────
accuracy    = accuracy_score(y_true, y_pred) * 100
macro_f1    = f1_score(y_true, y_pred, average="macro")
weighted_f1 = f1_score(y_true, y_pred, average="weighted")
kappa       = cohen_kappa_score(y_true, y_pred)

emotion_labels = [Config.EMOTION_NAMES[Config.REVERSE_MAP[i]] for i in range(5)]

print("\n" + "=" * 60)
print("  SENTIRA UC3 v2 — EEG EMOTION RECOGNITION")
print("  FINAL TEST SET EVALUATION (Unseen Subjects)")
print("=" * 60)
print(f"\n  Overall Accuracy : {accuracy:.2f}%")
print(f"  Macro-F1         : {macro_f1:.4f} ({macro_f1*100:.2f}%)")
print(f"  Weighted-F1      : {weighted_f1:.4f} ({weighted_f1*100:.2f}%)")
print(f"  Cohen's κ        : {kappa:.4f}")
print(f"\n  Test Samples     : {len(y_true)}")
print(f"  Test Subjects    : {len(split_info['test'])}")
print(f"  Chance level     : {100/Config.NUM_CLASSES:.1f}%")
print("=" * 60)
print("\n✅ Per-Class Classification Report:")
print(classification_report(y_true, y_pred, target_names=emotion_labels, digits=4))


  SENTIRA UC3 v2 — EEG EMOTION RECOGNITION
  FINAL TEST SET EVALUATION (Unseen Subjects)

  Overall Accuracy : 42.42%
  Macro-F1         : 0.4109 (41.09%)
  Weighted-F1      : 0.4109 (41.09%)
  Cohen's κ        : 0.2802

  Test Samples     : 2400
  Test Subjects    : 6
  Chance level     : 20.0%

✅ Per-Class Classification Report:
              precision    recall  f1-score   support

   Happiness     0.4950    0.6250    0.5525       480
     Sadness     0.3702    0.5437    0.4405       480
       Angry     0.6490    0.4083    0.5013       480
    Calmness     0.3472    0.1396    0.1991       480
     Neutral     0.3266    0.4042    0.3613       480

    accuracy                         0.4242      2400
   macro avg     0.4376    0.4242    0.4109      2400
weighted avg     0.4376    0.4242    0.4109      2400



## Cell 16 — CONFUSION MATRIX (Publication Quality 300 DPI)

In [18]:
import seaborn as sns

cm      = confusion_matrix(y_true, y_pred)
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)

fig, axes = plt.subplots(1, 2, figsize=(18, 7))
fig.suptitle(
    f"SENTIRA UC3 v2 — EEG Confusion Matrix | "
    f"Accuracy: {accuracy:.2f}% | κ={kappa:.4f}",
    fontsize=14, fontweight="bold"
)
ax = axes[0]
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=emotion_labels, yticklabels=emotion_labels, ax=ax,
            linewidths=0.5, linecolor="grey", cbar_kws={"label": "Count"})
ax.set_title("Raw Counts"); ax.set_xlabel("Predicted"); ax.set_ylabel("True")
ax.tick_params(axis="x", rotation=30)

ax = axes[1]
sns.heatmap(cm_norm * 100, annot=True, fmt=".1f", cmap="Blues",
            xticklabels=emotion_labels, yticklabels=emotion_labels, ax=ax,
            linewidths=0.5, linecolor="grey",
            vmin=0, vmax=100, cbar_kws={"label": "Recall (%)"})
ax.set_title("Normalized (Row = Recall %)"); ax.set_xlabel("Predicted")
ax.tick_params(axis="x", rotation=30)

plt.tight_layout()
cm_path = Config.DRIVE_FIGURES_DIR / "confusion_matrix_UC3_v2.png"
plt.savefig(cm_path, dpi=Config.FIGURE_DPI, bbox_inches="tight")
plt.show()
LOG.info(f"✅ Confusion matrix saved: {cm_path}")

10:45:10 | INFO | ✅ Confusion matrix saved: /content/drive/MyDrive/THESIS/EEG Results/Figures/eeg/confusion_matrix_UC3_v2.png


INFO:SENTIRA-EEG:✅ Confusion matrix saved: /content/drive/MyDrive/THESIS/EEG Results/Figures/eeg/confusion_matrix_UC3_v2.png


## Cell 17 — PER-EMOTION F1 BAR CHART (300 DPI)

In [19]:
from sklearn.metrics import classification_report

report = classification_report(y_true, y_pred, target_names=emotion_labels,
                                output_dict=True)
metrics = {
    "Precision": [report[e]["precision"] * 100 for e in emotion_labels],
    "Recall"   : [report[e]["recall"]    * 100 for e in emotion_labels],
    "F1-Score" : [report[e]["f1-score"]  * 100 for e in emotion_labels],
}

x      = np.arange(len(emotion_labels))
width  = 0.25
colors = ["#2196F3", "#4CAF50", "#FF9800"]

fig, ax = plt.subplots(figsize=(13, 6))
for i, (metric, vals) in enumerate(metrics.items()):
    bars = ax.bar(x + i * width, vals, width, label=metric,
                  color=colors[i], alpha=0.85, edgecolor="white")
    for bar in bars:
        h = bar.get_height()
        ax.text(bar.get_x() + bar.get_width() / 2, h + 0.5,
                f"{h:.1f}", ha="center", va="bottom", fontsize=7.5)

ax.set_xlabel("Emotion Class"); ax.set_ylabel("Score (%)")
ax.set_title(
    f"SENTIRA UC3 v2 — Per-Emotion Classification Metrics\n"
    f"Macro-F1: {macro_f1*100:.2f}% | Accuracy: {accuracy:.2f}%",
    fontsize=12, fontweight="bold")
ax.set_xticks(x + width); ax.set_xticklabels(emotion_labels)
ax.set_ylim(0, 115); ax.legend(); ax.grid(axis="y", alpha=0.3)
ax.axhline(y=70, color="red", linestyle="--", linewidth=1, alpha=0.5,
           label="EEG Target (70%)")

plt.tight_layout()
f1_path = Config.DRIVE_FIGURES_DIR / "per_emotion_metrics_UC3_v2.png"
plt.savefig(f1_path, dpi=Config.FIGURE_DPI, bbox_inches="tight")
plt.show()
LOG.info(f"✅ Per-emotion chart saved: {f1_path}")

10:45:11 | INFO | ✅ Per-emotion chart saved: /content/drive/MyDrive/THESIS/EEG Results/Figures/eeg/per_emotion_metrics_UC3_v2.png


INFO:SENTIRA-EEG:✅ Per-emotion chart saved: /content/drive/MyDrive/THESIS/EEG Results/Figures/eeg/per_emotion_metrics_UC3_v2.png


## Cell 18 — ATTENTION WEIGHTS VISUALIZATION

In [20]:
stream_labels = ["EEGNet\n(Raw EEG)", "DE Features\n(Band Energy)",
                 "PSD Features\n(Welch Power)"]

mean_attn_per_emotion = {}
for emo_idx in range(5):
    mask = y_true == emo_idx
    if mask.sum() > 0:
        mean_attn_per_emotion[emo_idx] = attention_weights[mask].mean(axis=0)

attn_matrix = np.array([
    mean_attn_per_emotion.get(i, np.zeros(3)) for i in range(5)
])

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
fig.suptitle("SENTIRA UC3 v2 — Stream Attention Weight Analysis",
             fontsize=13, fontweight="bold")

ax = axes[0]
im = ax.imshow(attn_matrix, cmap="YlOrRd", aspect="auto", vmin=0, vmax=1)
ax.set_xticks(range(3)); ax.set_xticklabels(stream_labels, fontsize=9)
ax.set_yticks(range(5)); ax.set_yticklabels(emotion_labels, fontsize=9)
ax.set_title("Mean Attention per Emotion × Stream")
plt.colorbar(im, ax=ax, label="Attention Weight")
for i in range(5):
    for j in range(3):
        ax.text(j, i, f"{attn_matrix[i,j]:.3f}", ha="center", va="center",
                fontsize=9, color="black" if attn_matrix[i,j] < 0.5 else "white")

ax = axes[1]
overall_mean = attention_weights.mean(axis=0)
bars = ax.bar(range(3), overall_mean * 100,
              color=["#2196F3", "#4CAF50", "#FF9800"], alpha=0.8, edgecolor="white")
ax.set_xticks(range(3)); ax.set_xticklabels(stream_labels, fontsize=9)
ax.set_ylabel("Mean Attention Weight (%)")
ax.set_title(f"Overall Mean Attention (n={len(y_true)} samples)")
ax.set_ylim(0, 70)
for bar in bars:
    h = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2, h + 1,
            f"{h:.1f}%", ha="center", va="bottom", fontsize=10)

plt.tight_layout()
attn_path = Config.DRIVE_FIGURES_DIR / "attention_weights_UC3_v2.png"
plt.savefig(attn_path, dpi=Config.FIGURE_DPI, bbox_inches="tight")
plt.show()
LOG.info(f"✅ Attention weights saved: {attn_path}")

10:45:12 | INFO | ✅ Attention weights saved: /content/drive/MyDrive/THESIS/EEG Results/Figures/eeg/attention_weights_UC3_v2.png


INFO:SENTIRA-EEG:✅ Attention weights saved: /content/drive/MyDrive/THESIS/EEG Results/Figures/eeg/attention_weights_UC3_v2.png


## Cell 19 — GRAD-CAM: EEGNet TEMPORAL SALIENCY (NEW IN v2)

In [21]:
# REVISION: Temporal saliency via Grad-CAM on the EEGNet branch.
# Shows which time points in the 5-second window are most informative
# for each emotion class — essential for thesis interpretability chapter.
#
# Method: Gradient-weighted Class Activation Mapping (Selvaraju et al., 2017)
# applied to the output feature map of EEGNet Block 2.
# ═══════════════════════════════════════════════════════════════════════════
class GradCAM_EEGNet:
    """
    Grad-CAM applied to EEGNet's block2 output.
    Produces temporal saliency maps [n_time_steps] per class.
    """
    def __init__(self, model: EEGAttentionFusionModel):
        self.model   = model
        self.grads   = None
        self.acts    = None
        self._hooks  = []

    def _register_hooks(self):
        def save_grad(g):
            self.grads = g
        def save_act(module, inp, out):
            self.acts = out
            out.register_hook(save_grad)

        h = self.model.eegnet_enc.block2.register_forward_hook(save_act)
        self._hooks.append(h)

    def _clear_hooks(self):
        for h in self._hooks:
            h.remove()
        self._hooks.clear()

    def compute(self, raw_eeg: torch.Tensor, de_feat: torch.Tensor,
                psd_feat: torch.Tensor, target_class: int) -> np.ndarray:
        """
        Returns saliency [n_time_steps] for a single sample and target class.
        """
        self._register_hooks()
        self.model.eval()

        raw_eeg  = raw_eeg.unsqueeze(0).requires_grad_(False).to(DEVICE)
        de_feat  = de_feat.unsqueeze(0).to(DEVICE)
        psd_feat = psd_feat.unsqueeze(0).to(DEVICE)

        logits = self.model(raw_eeg, de_feat, psd_feat)
        score  = logits[0, target_class]
        score.backward()

        # Global average pool over spatial dimensions
        grads    = self.grads[0]            # [C, 1, T]
        acts     = self.acts[0]             # [C, 1, T]
        weights  = grads.mean(dim=(1, 2))   # [C]
        cam      = (weights[:, None, None] * acts).sum(dim=0)   # [1, T]
        cam      = F.relu(cam).squeeze().cpu().detach().numpy()  # [T]
        # Normalise to [0, 1]
        if cam.max() > 0:
            cam = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)

        self._clear_hooks()
        return cam


# Compute Grad-CAM for one test sample per emotion class
grad_cam      = GradCAM_EEGNet(model)
time_axis     = np.linspace(0, Config.EPOCH_DURATION, model.eegnet_enc.block2(
    model.eegnet_enc.block1(
        torch.zeros(1, 1, Config.N_CHANNELS, Config.EPOCH_SAMPLES).to(DEVICE)
    )
).shape[-1])

fig, axes = plt.subplots(5, 1, figsize=(14, 12), sharex=True)
fig.suptitle("SENTIRA UC3 v2 — EEGNet Temporal Saliency (Grad-CAM per Emotion)",
             fontsize=13, fontweight="bold")
colors_emo = ["#E91E63", "#2196F3", "#FF9800", "#4CAF50", "#9C27B0"]

for emo_idx in range(5):
    # Find a test sample of this class
    emo_mask    = y_true == emo_idx
    sample_idxs = np.where(emo_mask)[0]
    if len(sample_idxs) == 0:
        continue
    s_idx = sample_idxs[0]

    raw_sample = torch.from_numpy(te_raw[s_idx]).float()
    de_sample  = torch.from_numpy(te_de[s_idx]).float()
    psd_sample = torch.from_numpy(te_psd[s_idx]).float()

    cam = grad_cam.compute(raw_sample, de_sample, psd_sample, emo_idx)

    ax = axes[emo_idx]
    ax.fill_between(time_axis, cam, alpha=0.6, color=colors_emo[emo_idx])
    ax.plot(time_axis, cam, color=colors_emo[emo_idx], linewidth=1.5)
    ax.set_ylabel("Saliency", fontsize=9)
    ax.set_title(f"{emotion_labels[emo_idx]}", fontsize=10, fontweight="bold")
    ax.set_ylim(0, 1.1); ax.grid(True, alpha=0.3)

axes[-1].set_xlabel("Time (seconds)")
plt.tight_layout()
gradcam_path = Config.DRIVE_FIGURES_DIR / "gradcam_temporal_UC3_v2.png"
plt.savefig(gradcam_path, dpi=Config.FIGURE_DPI, bbox_inches="tight")
plt.show()
LOG.info(f"✅ Grad-CAM figure saved: {gradcam_path}")

10:45:13 | INFO | ✅ Grad-CAM figure saved: /content/drive/MyDrive/THESIS/EEG Results/Figures/eeg/gradcam_temporal_UC3_v2.png


INFO:SENTIRA-EEG:✅ Grad-CAM figure saved: /content/drive/MyDrive/THESIS/EEG Results/Figures/eeg/gradcam_temporal_UC3_v2.png


## Cell 20 — STATISTICAL TESTS (McNemar + Cochran's Q)

In [22]:
# REVISION: McNemar test now checks for subject-level alignment before
# comparing UC1 (Audio) and UC3 (EEG) predictions.
# If the test sets differ in subjects, the test reports this clearly
# rather than silently skipping or crashing.
# ═══════════════════════════════════════════════════════════════════════════
import numpy as np
from pathlib import Path

PREDS_DRIVE_PATH = Config.DRIVE_RESULTS_DIR / "uc3_predictions_v2.npz"

np.savez(
    PREDS_DRIVE_PATH,
    y_true          = y_true,
    y_pred          = y_pred,
    y_probs         = y_probs_all,
    test_subjects   = np.array(split_info["test"], dtype=str),
    attention_weights = attention_weights,
)
LOG.info(f"✅ Predictions saved: {PREDS_DRIVE_PATH}")

def mcnemar_test(preds_a: np.ndarray, preds_b: np.ndarray,
                 y_true: np.ndarray, label: str = "McNemar"):
    """
    McNemar's test comparing two classifiers on the SAME test set.
    Uses continuity correction (Yates) for small samples.
    H0: Both classifiers have the same error rate.
    p < 0.05 → statistically significant difference.
    """
    from statsmodels.stats.contingency_tables import mcnemar
    correct_a = (preds_a == y_true)
    correct_b = (preds_b == y_true)
    n11 = (correct_a  & correct_b ).sum()
    n10 = (correct_a  & ~correct_b).sum()
    n01 = (~correct_a & correct_b ).sum()
    n00 = (~correct_a & ~correct_b).sum()
    table  = np.array([[n11, n10], [n01, n00]])
    result = mcnemar(table, exact=False, correction=True)
    sig    = result.pvalue < 0.05

    print(f"\n  {label}:")
    print(f"    Both correct   : {n11}")
    print(f"    Only A correct : {n10}")
    print(f"    Only B correct : {n01}")
    print(f"    Both wrong     : {n00}")
    print(f"    χ² statistic   : {result.statistic:.4f}")
    print(f"    p-value        : {result.pvalue:.4e} "
          f"{'✅ SIGNIFICANT' if sig else '  Not significant'}")
    return result

UC1_PREDS_PATH = (Config.DRIVE_ROOT /
                  "Audio Results/Results/audio/uc1_predictions.npz")

print("\n" + "=" * 60)
print("  UC3 vs UC1 McNemar Test (EEG vs Audio)")
print("=" * 60)

if UC1_PREDS_PATH.exists():
    uc1 = np.load(UC1_PREDS_PATH, allow_pickle=True)
    uc1_subjects = set(uc1["test_subjects"].tolist())
    uc3_subjects = set(split_info["test"])

    if uc1_subjects == uc3_subjects and len(uc1["y_pred"]) == len(y_pred):
        mcnemar_test(uc1["y_pred"], y_pred, y_true, "UC1(Audio) vs UC3(EEG)")
    else:
        print(f"\n  ⚠️  Test subject sets differ:")
        print(f"      UC1 subjects: {sorted(uc1_subjects)}")
        print(f"      UC3 subjects: {sorted(uc3_subjects)}")
        print(f"      McNemar requires IDENTICAL test sets.")
        print(f"      Run FusionEngine (UC4-UC7) with the shared test set.")
        print(f"      At inference time, align subjects: "
              f"shared = uc1_subjects ∩ uc3_subjects")
else:
    print(f"\n  UC1 predictions not found at: {UC1_PREDS_PATH}")
    print("  Run McNemar after completing UC1 (Audio) model.")
    print("  Template:")
    print("    uc1 = np.load(UC1_PREDS_PATH, allow_pickle=True)")
    print("    uc3 = np.load(PREDS_DRIVE_PATH, allow_pickle=True)")
    print("    mcnemar_test(uc1['y_pred'], uc3['y_pred'], uc1['y_true'])")

print("\n" + "=" * 60)
print("  STATISTICAL FRAMEWORK READY")
print("=" * 60)

10:45:13 | INFO | ✅ Predictions saved: /content/drive/MyDrive/THESIS/EEG Results/Results/eeg/uc3_predictions_v2.npz


INFO:SENTIRA-EEG:✅ Predictions saved: /content/drive/MyDrive/THESIS/EEG Results/Results/eeg/uc3_predictions_v2.npz



  UC3 vs UC1 McNemar Test (EEG vs Audio)

  ⚠️  Test subject sets differ:
      UC1 subjects: ['Subject1', 'Subject7', 'subject15', 'subject16', 'subject18', 'subject41']
      UC3 subjects: ['Subject15', 'Subject16', 'Subject18', 'Subject2', 'Subject41', 'Subject8']
      McNemar requires IDENTICAL test sets.
      Run FusionEngine (UC4-UC7) with the shared test set.
      At inference time, align subjects: shared = uc1_subjects ∩ uc3_subjects

  STATISTICAL FRAMEWORK READY


## Cell 21 — SAVE COMPLETE RESULTS (IEEE PAPER TABLE)

In [23]:
import json
from datetime import datetime

report_dict = classification_report(y_true, y_pred,
                                     target_names=emotion_labels,
                                     output_dict=True)

results = {
    "use_case"       : "UC3",
    "revision"       : "v2.0",
    "description"    : "EEG Only — EEGNet(D=2) + DE + PSD Attention Fusion",
    "dataset"        : "EAV",
    "n_test_samples" : int(len(y_true)),
    "n_test_subjects": len(split_info["test"]),
    "test_subjects"  : split_info["test"],
    "seed"           : Config.SEED,
    "evaluated_at"   : datetime.now().isoformat(),
    # ── Primary Metrics ──
    "accuracy"       : round(accuracy, 4),
    "macro_f1"       : round(macro_f1, 4),
    "weighted_f1"    : round(weighted_f1, 4),
    "cohen_kappa"    : round(kappa, 4),
    # ── Per-class ──
    "per_class": {
        emo: {
            "precision": round(report_dict[emo]["precision"], 4),
            "recall"   : round(report_dict[emo]["recall"],    4),
            "f1_score" : round(report_dict[emo]["f1-score"],  4),
            "support"  : int(report_dict[emo]["support"]),
        }
        for emo in emotion_labels
    },
    "confusion_matrix"           : cm.tolist(),
    "confusion_matrix_normalized": (cm.astype(float) /
                                    cm.sum(axis=1, keepdims=True)).round(4).tolist(),
    # ── Model Config ──
    "model": {
        "architecture" : "EEGAttentionFusionModel",
        "n_channels"   : Config.N_CHANNELS,
        "epoch_samples": Config.EPOCH_SAMPLES,
        "de_dim"       : Config.DE_DIM,
        "psd_dim"      : Config.PSD_DIM,
        "hidden_dim"   : Config.HIDDEN_DIM,
        "n_params"     : n_params,
        "eegnet_F1"    : Config.EEGNET_F1,
        "eegnet_D"     : Config.EEGNET_D,   # v2: D=2
        "eegnet_F2"    : Config.EEGNET_F2,
        "orig_sfreq"   : Config.ORIG_SFREQ,
        "target_sfreq" : Config.TARGET_SFREQ,
        "epoch_duration": Config.EPOCH_DURATION,
    },
    "mean_attention_weights": {
        "eegnet_raw"   : float(attention_weights[:, 0].mean()),
        "de_features"  : float(attention_weights[:, 1].mean()),
        "psd_features" : float(attention_weights[:, 2].mean()),
    },
    "split_note": "EEG-specific 30/6/6 split (independent of Audio UC1 split)",
}

RESULTS_PATH = Config.DRIVE_RESULTS_DIR / "uc3_results_v2.json"
with open(RESULTS_PATH, "w") as f:
    json.dump(results, f, indent=2)
LOG.info(f"✅ Results saved: {RESULTS_PATH}")

print("\n" + "=" * 70)
print("  IEEE PAPER — RESULTS TABLE (UC3 v2 Row)")
print("=" * 70)
print(f"  | UC3 | ✗ | ✗ | ✓ | {accuracy:.2f}% | {macro_f1:.4f} | {kappa:.4f} |")
print("=" * 70)
print("\n  Per-emotion F1 for thesis Table 4:")
for emo in emotion_labels:
    f1   = report_dict[emo]["f1-score"]
    prec = report_dict[emo]["precision"]
    rec  = report_dict[emo]["recall"]
    print(f"    {emo:<12}: F1={f1:.4f}  Prec={prec:.4f}  Rec={rec:.4f}")

10:45:15 | INFO | ✅ Results saved: /content/drive/MyDrive/THESIS/EEG Results/Results/eeg/uc3_results_v2.json


INFO:SENTIRA-EEG:✅ Results saved: /content/drive/MyDrive/THESIS/EEG Results/Results/eeg/uc3_results_v2.json



  IEEE PAPER — RESULTS TABLE (UC3 v2 Row)
  | UC3 | ✗ | ✗ | ✓ | 42.42% | 0.4109 | 0.2802 |

  Per-emotion F1 for thesis Table 4:
    Happiness   : F1=0.5525  Prec=0.4950  Rec=0.6250
    Sadness     : F1=0.4405  Prec=0.3702  Rec=0.5437
    Angry       : F1=0.5013  Prec=0.6490  Rec=0.4083
    Calmness    : F1=0.1991  Prec=0.3472  Rec=0.1396
    Neutral     : F1=0.3613  Prec=0.3266  Rec=0.4042


## Cell 22 — FUSION-READY INFERENCE WRAPPER

In [24]:
import pickle

class EEGFusionWrapper:
    """
    SENTIRA EEG Branch Inference Wrapper.
    Encapsulates: scalers + EEGAttentionFusionModel.

    Input:  raw EEG epoch [n_channels, n_samples_at_target_sfreq]
            = [30, 500] — pre-downsampled to 100 Hz, pre-filtered

    Usage in FusionEngine:
        eeg_wrapper = EEGFusionWrapper.load(drive_path, device)
        probs = eeg_wrapper.predict_proba(eeg_epoch)  # [5] tensor
        conf  = eeg_wrapper.get_confidence(eeg_epoch) # scalar
    """
    LABEL_MAP_INT = {0: "Happiness", 1: "Sadness", 2: "Angry",
                     3: "Calmness",  4: "Neutral"}

    def __init__(self, model: EEGAttentionFusionModel,
                 scalers: dict, device: torch.device,
                 config_snapshot: dict):
        self.model           = model.eval()
        self.scalers         = scalers
        self.device          = device
        self.config_snapshot = config_snapshot

    def _preprocess_epoch(self, eeg_epoch: np.ndarray
                          ) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
        epoch = eeg_epoch.astype(np.float32)
        # Epoch normalisation (same as training pipeline)
        mean  = epoch.mean(axis=1, keepdims=True)
        std   = epoch.std(axis=1, keepdims=True) + 1e-8
        epoch = (epoch - mean) / std
        # Spectral features
        sfreq    = self.config_snapshot.get("target_sfreq", 100)
        de_feat  = compute_de(epoch, sfreq)
        psd_feat = compute_psd(epoch, sfreq)
        # Scale
        de_scaled  = self.scalers["de"].transform(de_feat.reshape(1, -1)).astype(np.float32)
        psd_scaled = self.scalers["psd"].transform(psd_feat.reshape(1, -1)).astype(np.float32)
        # Tensors
        raw_t = torch.from_numpy(epoch[np.newaxis]).to(self.device)
        de_t  = torch.from_numpy(de_scaled).to(self.device)
        psd_t = torch.from_numpy(psd_scaled).to(self.device)
        return raw_t, de_t, psd_t

    @torch.no_grad()
    def predict_proba(self, eeg_epoch: np.ndarray) -> torch.Tensor:
        """Returns softmax probability vector [5]."""
        raw_t, de_t, psd_t = self._preprocess_epoch(eeg_epoch)
        return self.model.get_softmax_probs(raw_t, de_t, psd_t).squeeze(0)

    @torch.no_grad()
    def get_confidence(self, eeg_epoch: np.ndarray) -> float:
        return float(self.predict_proba(eeg_epoch).max().item())

    @torch.no_grad()
    def predict(self, eeg_epoch: np.ndarray) -> dict:
        probs      = self.predict_proba(eeg_epoch)
        pred_idx   = int(probs.argmax().item())
        confidence = float(probs.max().item())
        return {
            "predicted_class"  : pred_idx,
            "predicted_emotion": self.LABEL_MAP_INT[pred_idx],
            "confidence"       : confidence,
            "probabilities"    : {self.LABEL_MAP_INT[i]: float(probs[i].item())
                                  for i in range(5)},
        }

    def save(self, path: Path):
        state = {
            "model_state_dict": self.model.state_dict(),
            "model_config"    : {
                "n_channels" : self.model.n_channels,
                "n_samples"  : self.model.n_samples,
                "de_dim"     : self.model.de_dim,
                "psd_dim"    : self.model.psd_dim,
                "hidden_dim" : self.model.hidden_dim,
            },
            "scalers"         : self.scalers,
            "config_snapshot" : self.config_snapshot,
            "revision"        : "v2.0",
        }
        with open(path, "wb") as f:
            pickle.dump(state, f)
        LOG.info(f"✅ EEGFusionWrapper v2 saved: {path}")

    @classmethod
    def load(cls, path: Path, device: torch.device) -> "EEGFusionWrapper":
        with open(path, "rb") as f:
            state = pickle.load(f)
        m = EEGAttentionFusionModel(**state["model_config"]).to(device)
        m.load_state_dict(state["model_state_dict"])
        m.eval()
        return cls(model=m, scalers=state["scalers"],
                   device=device, config_snapshot=state["config_snapshot"])


config_snapshot = {
    "orig_sfreq"    : Config.ORIG_SFREQ,
    "target_sfreq"  : Config.TARGET_SFREQ,
    "epoch_duration": Config.EPOCH_DURATION,
    "epoch_samples" : Config.EPOCH_SAMPLES,
    "n_channels"    : Config.N_CHANNELS,
    "bandpass_low"  : Config.BANDPASS_LOW,
    "bandpass_high" : Config.BANDPASS_HIGH,
    "de_dim"        : Config.DE_DIM,
    "psd_dim"       : Config.PSD_DIM,
    "label_map"     : Config.LABEL_MAP,
    "eav_to_label"  : Config.EAV_TO_LABEL_MAP,
    "eegnet_D"      : Config.EEGNET_D,   # Record fix for auditability
}

eeg_wrapper       = EEGFusionWrapper(model, scalers, DEVICE, config_snapshot)
WRAPPER_DRIVE_PATH = Config.DRIVE_MODELS_DIR / "eeg_fusion_wrapper_v2.pkl"
eeg_wrapper.save(WRAPPER_DRIVE_PATH)

10:45:15 | INFO | ✅ EEGFusionWrapper v2 saved: /content/drive/MyDrive/THESIS/EEG Results/Models/eeg/eeg_fusion_wrapper_v2.pkl


INFO:SENTIRA-EEG:✅ EEGFusionWrapper v2 saved: /content/drive/MyDrive/THESIS/EEG Results/Models/eeg/eeg_fusion_wrapper_v2.pkl


## Cell 23 — VALIDATE FUSION WRAPPER

In [25]:
rng       = random.Random(Config.SEED + 1)
test_idx  = rng.randint(0, len(test_dataset) - 1)
raw_sample  = te_raw[test_idx]
true_label  = int(test_labels[test_idx])

print(f"  Test sample index : {test_idx}")
print(f"  True emotion      : {Config.EMOTION_NAMES[Config.REVERSE_MAP[true_label]]} "
      f"(label={true_label})")

result = eeg_wrapper.predict(raw_sample)
print(f"\n  Predicted Emotion : {result['predicted_emotion']}")
print(f"  Confidence        : {result['confidence']:.4f}")
print(f"\n  Full Probability Vector:")
for emo, prob in result["probabilities"].items():
    bar = "█" * int(prob * 40)
    print(f"    {emo:<12}: {prob:.4f} {bar}")

gate = result["confidence"] >= 0.3
print(f"\n  ✅ Confidence gate (threshold=0.3): "
      f"{'PASS — include in fusion' if gate else 'FAIL — exclude (low confidence)'}")

wrapper_reloaded   = EEGFusionWrapper.load(WRAPPER_DRIVE_PATH, DEVICE)
result_reloaded    = wrapper_reloaded.predict(raw_sample)
assert result["predicted_class"] == result_reloaded["predicted_class"], \
    "❌ Wrapper reload mismatch!"
print("\n  ✅ Wrapper reload test PASSED — serialize/deserialize verified.")

  Test sample index : 157
  True emotion      : Calmness (label=3)

  Predicted Emotion : Neutral
  Confidence        : 0.5831

  Full Probability Vector:
    Happiness   : 0.0166 
    Sadness     : 0.1366 █████
    Angry       : 0.0178 
    Calmness    : 0.2459 █████████
    Neutral     : 0.5831 ███████████████████████

  ✅ Confidence gate (threshold=0.3): PASS — include in fusion

  ✅ Wrapper reload test PASSED — serialize/deserialize verified.


## Cell 24 — FINAL DRIVE SYNC — ALL ARTIFACTS

In [26]:
from datetime import datetime

manifest = {
    "project"    : "SENTIRA",
    "use_case"   : "UC3 — EEG Only",
    "revision"   : "v2.0",
    "created"    : datetime.now().isoformat(),
    "key_fixes"  : [
        "BUG FIX: EEG now uses its OWN 30/6/6 split (was inheriting "
        "Audio split → only 3 train subjects → 1,200 training epochs)",
        "BUG FIX: EEGNet D reduced 8→2 (matches Lawhern 2018 paper)",
        "BUG FIX: label_smoothing removed from val/test criterion",
        "IMPROVEMENT: EEG data augmentation (noise, shift, ch-dropout)",
        "IMPROVEMENT: CosineAnnealingWarmRestarts LR schedule",
        "IMPROVEMENT: Grad-CAM temporal saliency per emotion",
        "IMPROVEMENT: McNemar test with subject alignment check",
    ],
    "files": {
        "hdf5_features"   : str(Config.DRIVE_FEATURES_DIR / "eeg_features.h5"),
        "best_model"      : str(MODEL_DRIVE_PATH),
        "scalers"         : str(SCALER_DRIVE_PATH),
        "subject_split"   : str(EEG_SPLIT_DRIVE),
        "training_history": str(HISTORY_DRIVE_PATH),
        "fusion_wrapper"  : str(WRAPPER_DRIVE_PATH),
        "results_json"    : str(RESULTS_PATH),
        "predictions_npz" : str(PREDS_DRIVE_PATH),
        "figure_cm"       : str(Config.DRIVE_FIGURES_DIR / "confusion_matrix_UC3_v2.png"),
        "figure_f1"       : str(Config.DRIVE_FIGURES_DIR / "per_emotion_metrics_UC3_v2.png"),
        "figure_attention": str(Config.DRIVE_FIGURES_DIR / "attention_weights_UC3_v2.png"),
        "figure_training" : str(Config.DRIVE_FIGURES_DIR / "training_curves_UC3_v2.png"),
        "figure_gradcam"  : str(Config.DRIVE_FIGURES_DIR / "gradcam_temporal_UC3_v2.png"),
    },
    "metrics": {
        "accuracy"    : round(accuracy, 4),
        "macro_f1"    : round(macro_f1, 4),
        "weighted_f1" : round(weighted_f1, 4),
        "cohen_kappa" : round(kappa, 4),
    },
    "model_info": {
        "architecture"       : "EEGAttentionFusionModel",
        "streams"            : ["EEGNet(raw EEG, D=2)", "DE features", "PSD features"],
        "n_params"           : n_params,
        "target_sfreq_hz"   : Config.TARGET_SFREQ,
        "epoch_duration_sec" : Config.EPOCH_DURATION,
        "n_channels"         : Config.N_CHANNELS,
        "train_subjects"     : len(split_info["train"]),
        "val_subjects"       : len(split_info["val"]),
        "test_subjects"      : len(split_info["test"]),
    }
}

MANIFEST_PATH = Config.DRIVE_RESULTS_DIR / "uc3_manifest_v2.json"
with open(MANIFEST_PATH, "w") as f:
    json.dump(manifest, f, indent=2)

LOG.info("\n" + "=" * 65)
LOG.info("  ✅ SENTIRA UC3 v2 — COMPLETE. All artifacts saved to Drive.")
LOG.info("=" * 65)
for k, v in manifest["files"].items():
    exists = Path(v).exists()
    mark   = "✅" if exists else "⚠️ "
    LOG.info(f"  {mark} {k:<22}: {Path(v).name}")
LOG.info("\n  Final Metrics:")
for k, v in manifest["metrics"].items():
    LOG.info(f"    {k:<15}: {v}")
LOG.info("\n  Next step → FusionEngine (UC4-UC7):")
LOG.info("    Load: audio_fusion_wrapper.pkl, video_fusion_wrapper.pkl,")
LOG.info("           eeg_fusion_wrapper_v2.pkl")
LOG.info("    Align test subjects at inference time (not at split creation).")
LOG.info("    Implement: P_fused = Σ(w_i × P_i) / Σw_i")
LOG.info("=" * 65)

10:45:16 | INFO | 


INFO:SENTIRA-EEG:


10:45:16 | INFO |   ✅ SENTIRA UC3 v2 — COMPLETE. All artifacts saved to Drive.


INFO:SENTIRA-EEG:  ✅ SENTIRA UC3 v2 — COMPLETE. All artifacts saved to Drive.


10:45:16 | INFO | =================================================================


...

INFO:SENTIRA-EEG:    Align test subjects at inference time (not at split creation).


10:45:16 | INFO |     Implement: P_fused = Σ(w_i × P_i) / Σw_i


INFO:SENTIRA-EEG:    Implement: P_fused = Σ(w_i × P_i) / Σw_i


10:45:16 | INFO | =================================================================


INFO:SENTIRA-EEG:=================================================================
